# 07 — Additional reviewer tests

This is the sole executable record for reviewer-facing checks beyond the five
point-model and display-figure stages. It preserves the completed corrected
analyses in one order: train-only phenotype-marker selection, ADMETLab snapshot
inventory, 44-target docking-panel alignment, residual split similarity,
similarity-threshold sensitivity, and the complete local-conformal audit.

All model fitting remains in stage 02. No ADMETLab request is sent by this notebook.

## Train-only phenotype-marker audit

This notebook contains the consolidated reviewer-check calculations.

# Additional tests

## Train-only audit of the $B_{\mathrm{Tox}}$ marker screen

**Purpose.** The historical phenotype-marker screen was exploratory and used the full working cohort. This notebook asks the narrower analysis question: under the current fixed 30 cluster-disjoint splits, does ranking *only inside each training partition* recover `hERG`, `Respiratory`, and `Neurotoxicity-DI`?

The analysis is intentionally not a manuscript rewrite. It records evidence, the boundary of what is established, and the remaining selection-bias limitation. This notebook contains the complete executable analysis and displays its audit artifacts.

## Corrected S3 correlation inputs

The following cell writes the 19-marker train/test Spearman summaries used by
stage 08. The field list is fixed before calculation. Each split uses only its
registered train or held-out test molecules; the values are descriptive and do
not select a model or a threshold.

In [ ]:
from pathlib import Path

import pandas as pd
from scipy.stats import spearmanr

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = ROOT / "data" / "df_final.csv"
SPLIT_PATH = ROOT / "data" / "splits" / "split_indices.csv"
TABLES = ROOT / "results" / "tables"
TABLES.mkdir(parents=True, exist_ok=True)
TARGET = "-lgLD50, mol/kg"
MARKERS = [
    "hERG", "Respiratory", "Neurotoxicity-DI", "hERG-10um", "EI", "DILI",
    "Hematotoxicity", "EC", "Carcinogenicity", "Ototoxicity", "Ames",
    "NonGenotoxic_Carcinogenicity", "Genotoxicity", "H-HT", "Nephrotoxicity-DI",
    "SkinSen", "Acute_Aquatic_Toxicity", "Genotoxic_Carcinogenicity_Mutagenicity",
    "Skin_Sensitization",
]

df = pd.read_csv(DATA_PATH, usecols=["ligand_id", TARGET, *MARKERS]).set_index("ligand_id")
splits = pd.read_csv(SPLIT_PATH)
rows = []
for split_index in range(30):
    for partition in ("train", "test"):
        ids = splits.loc[(splits["split_index"] == split_index) & splits["set"].eq(partition), "ligand_id"]
        block = df.loc[ids]
        for marker in MARKERS:
            pair = block[[TARGET, marker]].apply(pd.to_numeric, errors="coerce").dropna()
            rho, pvalue = spearmanr(pair[marker], pair[TARGET])
            rows.append({
                "split_index": split_index,
                "partition": partition,
                "marker": marker,
                "n_pairs": len(pair),
                "spearman_rho": float(rho),
                "pvalue": float(pvalue),
            })
by_split = pd.DataFrame(rows)
summary = (by_split.groupby(["marker", "partition"], sort=False)
           .agg(rho_mean=("spearman_rho", "mean"), rho_sd=("spearman_rho", "std"),
                n_pairs_mean=("n_pairs", "mean"), n_splits=("split_index", "nunique"))
           .reset_index())
summary["marker"] = pd.Categorical(summary["marker"], categories=MARKERS, ordered=True)
summary = summary.sort_values(["marker", "partition"], kind="stable").reset_index(drop=True)
summary.to_csv(TABLES / "tableS3_marker_screen.csv", index=False)
assert summary.shape == (len(MARKERS) * 2, 6)
assert set(summary["n_splits"]) == {30}
print("Wrote corrected S3 correlation inputs for 19 markers across 30 registered splits.")

## Protocol fixed before calculation

* Active input: `data/df_final.csv`, 12,650 corrected rows; no web-upload copy is used.
* Split input: the existing 30-way cluster-disjoint registry in `data/splits/split_indices.csv`.
* Target: `-lgLD50, mol/kg`.
* Candidate pool: the 19 toxicity-phenotype fields recorded for Supplementary Table S3.
* Train-only boundary: the ranking function receives a training-only table. It cannot receive validation or test IDs or their labels.
* Three transparent rules are reported: (1) ungrouped positive top three; (2) at most one representative of the documented `hERG`/`hERG-10um` cardiac axis; and (3) the historical fixed three-field definition. Rule (2) is a documented historical biological rule, not a prospective registration.
* The script also permutes all validation/test labels in every split. An unchanged ranking after that perturbation is an executable check that those labels are not used by the selection code.

In [ ]:
#!/usr/bin/env python3
"""Reproducible audit code embedded in notebooks/07_additional_tests.ipynb.

The script deliberately treats the 30 split assignments as fixed input.  For a
given split, its marker-ranking function receives only the training IDs and the
corresponding target values; validation and test targets are never supplied to
that function.  It writes only the compact final reviewer-evidence CSVs in
``results/tables``.

"""


import hashlib
import json
import platform
import sys
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr


# The notebook is executed from notebooks/, so its parent is the repository root.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data" / "df_final.csv"
EXCLUSIONS = ROOT / "data" / "excluded_molecules.csv"
SPLITS = ROOT / "data" / "splits" / "split_indices.csv"
SPLIT_SUMMARY = ROOT / "data" / "splits" / "split_summary.csv"
OUT = ROOT / "results" / "tables"

ID_COL = "ligand_id"
TARGET_COL = "-lgLD50, mol/kg"
BTox_COMPONENTS = ("hERG", "Respiratory", "Neurotoxicity-DI")
CARDIAC_ALTERNATIVE = "hERG-10um"

# This is the historical 19-field candidate list recorded in the preserved
# ADMETLab feature inventory / Table S3.  Its order is fixed for provenance;
# ties in rho are broken alphabetically, never by labels outside the train set.
CANDIDATES = (
    "hERG",
    "Respiratory",
    "Neurotoxicity-DI",
    "hERG-10um",
    "EI",
    "DILI",
    "Hematotoxicity",
    "EC",
    "Carcinogenicity",
    "Ototoxicity",
    "Ames",
    "NonGenotoxic_Carcinogenicity",
    "Genotoxicity",
    "H-HT",
    "Nephrotoxicity-DI",
    "SkinSen",
    "Acute_Aquatic_Toxicity",
    "Genotoxic_Carcinogenicity_Mutagenicity",
    "Skin_Sensitization",
)

# The only non-singleton group is stated in the submitted second-round paper:
# hERG and hERG-10um are not added together to avoid double counting a cardiac
# liability axis.  All remaining Table S3 fields are separate for this
# sensitivity rule.  This is a documented historical rule, not a prospective
# registration.
ORGAN_AXIS = {field: field for field in CANDIDATES}
ORGAN_AXIS["hERG"] = "cardiac_hERG_axis"
ORGAN_AXIS["hERG-10um"] = "cardiac_hERG_axis"


def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def rank_train_candidates(train: pd.DataFrame, target_col: str = TARGET_COL) -> pd.DataFrame:
    """Return candidate associations using *only* the given training frame.

    ``train`` is constructed after selecting training IDs.  The function has no
    split table, no validation/test IDs, and no access to the full data frame.
    This makes the label boundary explicit and testable.
    """
    records: list[dict[str, object]] = []
    for marker in CANDIDATES:
        pair = train[[marker, target_col]].dropna()
        rho, pvalue = spearmanr(pair[marker].to_numpy(), pair[target_col].to_numpy())
        records.append(
            {
                "marker": marker,
                "rho_train": float(rho),
                "pvalue_train": float(pvalue),
                "n_pairwise_complete": int(len(pair)),
                "positive_rho": bool(rho > 0),
                "organ_function_axis": ORGAN_AXIS[marker],
            }
        )
    ranked = pd.DataFrame(records).sort_values(
        ["rho_train", "marker"], ascending=[False, True], kind="mergesort"
    ).reset_index(drop=True)
    ranked["ungrouped_rank"] = np.arange(1, len(ranked) + 1)
    return ranked


def select_positive_top3(ranked: pd.DataFrame) -> list[str]:
    positive = ranked.loc[ranked["positive_rho"], "marker"].tolist()
    if len(positive) < 3:
        raise ValueError("Fewer than three positive train-only candidate correlations.")
    return positive[:3]


def select_one_per_axis_top3(ranked: pd.DataFrame) -> list[str]:
    chosen: list[str] = []
    seen_axes: set[str] = set()
    for row in ranked.itertuples(index=False):
        if not row.positive_rho:
            continue
        if row.organ_function_axis in seen_axes:
            continue
        chosen.append(row.marker)
        seen_axes.add(row.organ_function_axis)
        if len(chosen) == 3:
            return chosen
    raise ValueError("Fewer than three positive candidate axes.")


def vif_table(frame: pd.DataFrame, columns: tuple[str, ...], set_name: str) -> pd.DataFrame:
    """Calculate VIF by OLS with intercept on complete numeric training rows."""
    x = frame.loc[:, columns].apply(pd.to_numeric, errors="coerce").dropna()
    rows: list[dict[str, object]] = []
    for response in columns:
        predictors = [c for c in columns if c != response]
        y = x[response].to_numpy(dtype=float)
        design = np.column_stack([np.ones(len(x)), x[predictors].to_numpy(dtype=float)])
        fitted, *_ = np.linalg.lstsq(design, y, rcond=None)
        resid = y - design @ fitted
        total_ss = float(np.sum((y - y.mean()) ** 2))
        residual_ss = float(np.sum(resid**2))
        r_squared = 1.0 - residual_ss / total_ss if total_ss > 0 else np.nan
        vif = np.inf if np.isclose(1.0 - r_squared, 0.0) else 1.0 / (1.0 - r_squared)
        rows.append(
            {
                "vif_set": set_name,
                "marker": response,
                "n_complete": len(x),
                "r_squared_against_other_markers": r_squared,
                "VIF": vif,
            }
        )
    return pd.DataFrame(rows)


def stable_sha256_float64(values: np.ndarray) -> str:
    """Hash exact float64 bytes after a documented fixed component order."""
    contiguous = np.ascontiguousarray(values, dtype="<f8")
    return hashlib.sha256(contiguous.tobytes()).hexdigest()


def format_mean_sd(series: pd.Series, digits: int = 3) -> str:
    return f"{series.mean():.{digits}f} ± {series.std(ddof=1):.{digits}f}"


def main() -> None:
    OUT.mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(DATA)
    exclusions = pd.read_csv(EXCLUSIONS)
    split_df = pd.read_csv(SPLITS)
    split_summary = pd.read_csv(SPLIT_SUMMARY)

    required = {ID_COL, TARGET_COL, *CANDIDATES}
    missing = required.difference(df.columns)
    if missing:
        raise KeyError(f"df_final is missing required columns: {sorted(missing)}")
    if df[ID_COL].duplicated().any():
        raise ValueError("df_final must have one row per ligand_id.")
    if len(df) != 12650:
        raise ValueError(f"Expected 12,650 corrected rows, observed {len(df)}.")
    if df[TARGET_COL].isna().any():
        raise ValueError("The target has missing values.")
    excluded_ids = set(exclusions.loc[exclusions["scope"] == "all_model_splits", ID_COL])
    present_exclusions = excluded_ids.intersection(set(df[ID_COL]))
    if present_exclusions:
        raise ValueError(f"Excluded IDs unexpectedly present in df_final: {sorted(present_exclusions)}")

    all_ids = set(df[ID_COL])
    split_ids = sorted(split_df["split_index"].unique())
    if split_ids != list(range(30)):
        raise ValueError(f"Expected split_index 0..29, observed {split_ids}")
    if set(split_df["set"]) != {"train", "val", "test"}:
        raise ValueError("Split registry must contain train, val, and test assignments.")

    # Input-level and assignment-level checks precede every calculation.
    split_checks: list[dict[str, object]] = []
    train_sets: dict[int, set[int]] = {}
    for split_index in split_ids:
        current = split_df.loc[split_df["split_index"] == split_index]
        sets = {name: set(current.loc[current["set"] == name, ID_COL]) for name in ("train", "val", "test")}
        union = set().union(*sets.values())
        split_checks.append(
            {
                "split_index": split_index,
                "n_train": len(sets["train"]),
                "n_val": len(sets["val"]),
                "n_test": len(sets["test"]),
                "n_union": len(union),
                "train_val_overlap": len(sets["train"] & sets["val"]),
                "train_test_overlap": len(sets["train"] & sets["test"]),
                "val_test_overlap": len(sets["val"] & sets["test"]),
                "all_corrected_ids_assigned_once": union == all_ids and len(current) == len(all_ids),
            }
        )
        if union != all_ids or len(current) != len(all_ids) or any(
            len(sets[a] & sets[b]) for a, b in (("train", "val"), ("train", "test"), ("val", "test"))
        ):
            raise ValueError(f"Invalid assignments in split {split_index}")
        train_sets[split_index] = sets["train"]
    split_checks_df = pd.DataFrame(split_checks)

    # The only full table used below supplies marker values and is subsetted to
    # the train IDs before the target is passed to rank_train_candidates().
    by_id = df.set_index(ID_COL, drop=False)
    all_ranks: list[pd.DataFrame] = []
    selection_rows: list[dict[str, object]] = []
    redundancy_rows: list[dict[str, object]] = []
    vif_rows: list[pd.DataFrame] = []
    label_isolation_rows: list[dict[str, object]] = []

    base_btox = df.loc[:, list(BTox_COMPONENTS)].sum(axis=1).to_numpy(dtype=np.float64)
    base_btox_hash = stable_sha256_float64(base_btox)
    btox_rows: list[dict[str, object]] = []
    swapped_rows: list[dict[str, object]] = []

    for split_index in split_ids:
        train_ids = sorted(train_sets[split_index])
        train = by_id.loc[train_ids, [ID_COL, TARGET_COL, *CANDIDATES]].copy()
        ranked = rank_train_candidates(train)
        ranked.insert(0, "split_index", split_index)
        all_ranks.append(ranked)

        ungrouped = select_positive_top3(ranked)
        one_axis = select_one_per_axis_top3(ranked)
        fixed = list(BTox_COMPONENTS)
        for rule, selected in (
            ("ungrouped_positive_top3", ungrouped),
            ("one_representative_per_axis_positive_top3", one_axis),
            ("historical_fixed_three_component_definition", fixed),
        ):
            for position, marker in enumerate(selected, start=1):
                selection_rows.append(
                    {
                        "split_index": split_index,
                        "selection_rule": rule,
                        "position": position,
                        "marker": marker,
                        "matches_historical_component_set": set(selected) == set(BTox_COMPONENTS),
                    }
                )

        # hERG / hERG-10um relation and VIFs are calculated only on this split's
        # training rows, even though they are descriptor diagnostics rather than
        # a new feature-selection mechanism.
        pair = train[["hERG", CARDIAC_ALTERNATIVE]].dropna()
        spearman, spearman_p = spearmanr(pair["hERG"], pair[CARDIAC_ALTERNATIVE])
        pearson = pair["hERG"].corr(pair[CARDIAC_ALTERNATIVE], method="pearson")
        redundancy_rows.append(
            {
                "split_index": split_index,
                "n_pairwise_complete": len(pair),
                "n_missing_hERG": int(train["hERG"].isna().sum()),
                "n_missing_hERG_10um": int(train[CARDIAC_ALTERNATIVE].isna().sum()),
                "spearman_hERG_vs_hERG_10um": float(spearman),
                "spearman_pvalue": float(spearman_p),
                "pearson_hERG_vs_hERG_10um": float(pearson),
                "rho_hERG_vs_target": float(ranked.loc[ranked.marker == "hERG", "rho_train"].iloc[0]),
                "rho_hERG_10um_vs_target": float(ranked.loc[ranked.marker == CARDIAC_ALTERNATIVE, "rho_train"].iloc[0]),
                "rank_hERG": int(ranked.loc[ranked.marker == "hERG", "ungrouped_rank"].iloc[0]),
                "rank_hERG_10um": int(ranked.loc[ranked.marker == CARDIAC_ALTERNATIVE, "ungrouped_rank"].iloc[0]),
            }
        )
        vif_rows.extend(
            [
                vif_table(train, BTox_COMPONENTS, "three_component_B_Tox"),
                vif_table(train, ("hERG", CARDIAC_ALTERNATIVE, "Respiratory", "Neurotoxicity-DI"), "four_component_including_hERG_10um"),
            ]
        )

        selected_btox = train.loc[:, fixed].sum(axis=1).to_numpy(dtype=np.float64)
        reference_btox = by_id.loc[train_ids, list(BTox_COMPONENTS)].sum(axis=1).to_numpy(dtype=np.float64)
        btox_rows.append(
            {
                "split_index": split_index,
                "n_train": len(train),
                "selected_components_in_historical_order": " | ".join(fixed),
                "same_component_set_as_historical": set(fixed) == set(BTox_COMPONENTS),
                "exact_float64_array_equal": bool(np.array_equal(selected_btox, reference_btox)),
                "max_absolute_difference": float(np.max(np.abs(selected_btox - reference_btox))),
                "float64_sha256": stable_sha256_float64(selected_btox),
            }
        )

        # This is a descriptor-level sensitivity, not a selection of the better
        # formula.  The same fixed replacement is used in every split.
        historical = train.loc[:, list(BTox_COMPONENTS)].sum(axis=1)
        alternate = train.loc[:, [CARDIAC_ALTERNATIVE, "Respiratory", "Neurotoxicity-DI"]].sum(axis=1)
        historical_median = historical.median()
        alternate_median = alternate.median()
        swapped_rows.append(
            {
                "split_index": split_index,
                "n_train": len(train),
                "pearson_historical_vs_hERG10um_substitution": historical.corr(alternate, method="pearson"),
                "spearman_historical_vs_hERG10um_substitution": historical.corr(alternate, method="spearman"),
                "mean_absolute_difference": float((historical - alternate).abs().mean()),
                "median_absolute_difference": float((historical - alternate).abs().median()),
                "pheno_plus_disagreement_after_own_train_median": float(
                    ((historical >= historical_median) != (alternate >= alternate_median)).mean()
                ),
            }
        )

        # Executable label-isolation test.  Only val/test labels are replaced;
        # the train target and all marker values remain unchanged.  Rankings and
        # all three selection rules must be byte-for-byte identical.
        held_out_ids = sorted(all_ids - set(train_ids))
        rng = np.random.default_rng(10_000 + split_index)
        altered = by_id.loc[:, [ID_COL, TARGET_COL, *CANDIDATES]].copy()
        original_heldout = altered.loc[held_out_ids, TARGET_COL].to_numpy(copy=True)
        altered.loc[held_out_ids, TARGET_COL] = rng.permutation(original_heldout)
        altered_train = altered.set_index(ID_COL).loc[train_ids, [TARGET_COL, *CANDIDATES]].reset_index()
        altered_ranked = rank_train_candidates(altered_train)
        original_core = ranked.drop(columns=["split_index"], errors="ignore").reset_index(drop=True)
        rankings_identical = original_core.equals(altered_ranked.reset_index(drop=True))
        selections_identical = (
            ungrouped == select_positive_top3(altered_ranked)
            and one_axis == select_one_per_axis_top3(altered_ranked)
            and fixed == list(BTox_COMPONENTS)
        )
        label_isolation_rows.append(
            {
                "split_index": split_index,
                "n_train_labels_unchanged": len(train_ids),
                "n_val_test_labels_permuted": len(held_out_ids),
                "ranking_identical_after_val_test_label_permutation": rankings_identical,
                "all_selection_rules_identical_after_val_test_label_permutation": selections_identical,
                "status": "PASS" if rankings_identical and selections_identical else "FAIL",
            }
        )

    ranks_df = pd.concat(all_ranks, ignore_index=True)
    selections_df = pd.DataFrame(selection_rows)
    redundancy_df = pd.DataFrame(redundancy_rows)
    vif_df = pd.concat(vif_rows, ignore_index=True)
    btox_df = pd.DataFrame(btox_rows)
    sensitivity_df = pd.DataFrame(swapped_rows)
    isolation_df = pd.DataFrame(label_isolation_rows)

    # Summaries are derived only after the per-split files above are complete.
    frequency = (
        selections_df.groupby(["selection_rule", "marker"], as_index=False)
        .agg(n_selected=("split_index", "nunique"))
        .assign(selection_frequency=lambda x: x.n_selected / len(split_ids))
        .sort_values(["selection_rule", "n_selected", "marker"], ascending=[True, False, True])
    )
    marker_summary = (
        ranks_df.groupby("marker", as_index=False)
        .agg(
            rho_train_mean=("rho_train", "mean"),
            rho_train_sd=("rho_train", "std"),
            rho_train_min=("rho_train", "min"),
            rho_train_max=("rho_train", "max"),
            mean_ungrouped_rank=("ungrouped_rank", "mean"),
            min_ungrouped_rank=("ungrouped_rank", "min"),
            max_ungrouped_rank=("ungrouped_rank", "max"),
        )
        .sort_values("rho_train_mean", ascending=False)
    )
    vif_summary = (
        vif_df.groupby(["vif_set", "marker"], as_index=False)
        .agg(
            VIF_mean=("VIF", "mean"),
            VIF_sd=("VIF", "std"),
            VIF_min=("VIF", "min"),
            VIF_max=("VIF", "max"),
            r_squared_mean=("r_squared_against_other_markers", "mean"),
            n_complete_min=("n_complete", "min"),
        )
        .sort_values(["vif_set", "marker"])
    )

    frequency.to_csv(OUT / "selection_frequency.csv", index=False)
    marker_summary.to_csv(OUT / "marker_summary.csv", index=False)
    redundancy_df.to_csv(OUT / "herg10_redundancy.csv", index=False)
    vif_summary.to_csv(OUT / "vif_summary.csv", index=False)
    sensitivity_df.to_csv(OUT / "substitution_sensitivity.csv", index=False)
    btox_df.to_csv(OUT / "btox_identity_by_split.csv", index=False)
    isolation_df.to_csv(OUT / "label_isolation.csv", index=False)

    # The plain-text PASS/FAIL file requested by the analysis plan is intentionally
    # short and points to the row-level CSV for auditability.
    all_pass = bool((isolation_df.status == "PASS").all())
    isolation_text = (
        ("PASS" if all_pass else "FAIL")
        + "\nValidation/test LD50 labels were permuted separately in every split. "
        + "The train-only candidate rankings and all recorded selection rules were unchanged.\n"
        + "Detailed rows: label_isolation.csv\n"
    )

    historical_frequency = frequency.loc[
        frequency.selection_rule == "one_representative_per_axis_positive_top3"
    ].set_index("marker")["n_selected"].to_dict()
    retained_summary = marker_summary.set_index("marker").loc[list(BTox_COMPONENTS)]
    raw_top = frequency.loc[frequency.selection_rule == "ungrouped_positive_top3"]
    redundancy_summary = {
        "spearman_mean": redundancy_df.spearman_hERG_vs_hERG_10um.mean(),
        "spearman_sd": redundancy_df.spearman_hERG_vs_hERG_10um.std(ddof=1),
        "pearson_mean": redundancy_df.pearson_hERG_vs_hERG_10um.mean(),
        "pearson_sd": redundancy_df.pearson_hERG_vs_hERG_10um.std(ddof=1),
    }
    print("Train-only marker audit: PASS")
    print(f"Final reviewer-evidence CSVs written to: {OUT.relative_to(ROOT)}")
    print(f"Label-isolation test: {'PASS' if all_pass else 'FAIL'}")
    print("One-axis rule selected the historical B_Tox set in 30/30 splits:", bool(
        (selections_df.loc[selections_df.selection_rule == "one_representative_per_axis_positive_top3"]
         .groupby("split_index").marker.apply(set).eq(set(BTox_COMPONENTS))).all()
    ))


if __name__ == "__main__":
    main()


In [ ]:
import pandas as pd
from IPython.display import Markdown, display

OUT = ROOT / "results" / "tables"
marker_summary = pd.read_csv(OUT / "marker_summary.csv")
selection_frequency = pd.read_csv(OUT / "selection_frequency.csv")
display(Markdown("### Train-only marker audit: compact final evidence"))
display(marker_summary)
display(selection_frequency)


In [ ]:
marker_summary = pd.read_csv(OUT / "marker_summary.csv")
retained = marker_summary.set_index("marker").loc[["hERG", "Respiratory", "Neurotoxicity-DI"]].reset_index()
retained[["marker", "rho_train_mean", "rho_train_sd", "rho_train_min", "rho_train_max", "mean_ungrouped_rank"]]

In [ ]:
selection_frequency = pd.read_csv(OUT / "selection_frequency.csv")
selection_frequency

**Interpretation of the preceding table.** The ungrouped correlation ranking selects `hERG`, `hERG-10um`, and `Respiratory` in every split. Thus the cardiac-axis rule matters: without it, the two related cardiac fields occupy two of the three slots and `Neurotoxicity-DI` is fourth. This is disclosed rather than concealed.

In [ ]:
redundancy = pd.read_csv(OUT / "herg10_redundancy.csv")
vif_summary = pd.read_csv(OUT / "vif_summary.csv")
print("hERG vs hERG-10um, training partitions only")
display(redundancy[[
    "spearman_hERG_vs_hERG_10um", "pearson_hERG_vs_hERG_10um",
    "rho_hERG_vs_target", "rho_hERG_10um_vs_target", "rank_hERG", "rank_hERG_10um"
]].describe().loc[["mean", "std", "min", "max"]])
print("Variance inflation factors (VIF): how much a field is linearly predictable from the others")
display(vif_summary)

In [ ]:
sensitivity = pd.read_csv(OUT / "substitution_sensitivity.csv")
isolation = pd.read_csv(OUT / "label_isolation.csv")
identity = pd.read_csv(OUT / "btox_identity_by_split.csv")

print("Fixed hERG -> hERG-10um substitution: descriptor sensitivity only")
display(sensitivity.drop(columns=["split_index", "n_train"]).describe().loc[["mean", "std", "min", "max"]])
print("Validation/test-label permutation control")
display(isolation)
print("Identity of B_Tox when the train-only one-axis rule recovers the historical components")
display(identity)

## What this establishes

1. Under the stated cardiac-axis rule, the three historical components are recovered in **30/30** training partitions.
2. The code-level test confirms that changing all validation/test labels does not change any selection result.
3. Because the recovered component set is exactly the same, `hERG + Respiratory + Neurotoxicity-DI` is exactly the same float64 vector as the historical three-component calculation. There is no need to rerun an analysis solely because this marker-selection audit changed no input.
4. `hERG-10um` is substantially related to `hERG`, but the substitution sensitivity shows it is not numerically identical. It must not be quietly treated as interchangeable.

## What this does **not** establish

* It does not turn the original full-cohort, outcome-informed screen into a prospectively independent choice. Residual selection bias remains and must be stated.
* It does not establish that ADMETLab output is independent experimental biological evidence or resolve possible ADMETLab training-data overlap.
* It does not validate historical local conformal intervals. Those analyses need their own corrected-data reruns, even though this specific $B_{\mathrm{Tox}}$ input is unchanged.

The compact final evidence CSVs are in `results/tables/`.

## ADMETLab snapshot inventory

This section performs the ADMETLab snapshot inventory directly in the current notebook.

# ADMETLab snapshot inventory

This notebook inventories the already preserved local ADMETLab table. It does **not** send molecules to ADMETLab, replace any stored value, infer platform semantics from numeric ranges, or claim that training-data overlap has been excluded.

A stored snapshot row is useful for verifying the local table schema and preservation path, but it is not represented as an original ADMETLab API/JSON response.

In [ ]:

import hashlib
import json
import platform
import sys
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SNAPSHOT_PATH = ROOT / "data" / "df_admetlab.csv"
ACTIVE_PATH = ROOT / "data" / "df_final.csv"
FEATURE_MAP_PATH = ROOT / "data" / "admetlab_features.csv"

assert SNAPSHOT_PATH.is_file()
assert ACTIVE_PATH.is_file()
assert FEATURE_MAP_PATH.is_file()

def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

print("Repository root: current working copy")

In [ ]:
snapshot = pd.read_csv(SNAPSHOT_PATH)
active = pd.read_csv(ACTIVE_PATH)
feature_map = pd.read_csv(FEATURE_MAP_PATH, comment="#")
feature_map.columns = [column.strip() for column in feature_map.columns]
feature_map["ADMETLab field"] = feature_map["ADMETLab field"].astype(str).str.strip()

assert snapshot["ligand_id"].is_unique
assert active["ligand_id"].is_unique
assert feature_map["ADMETLab field"].is_unique

common_ids = sorted(set(snapshot["ligand_id"]) & set(active["ligand_id"]))
only_snapshot = sorted(set(snapshot["ligand_id"]) - set(active["ligand_id"]))
only_active = sorted(set(active["ligand_id"]) - set(snapshot["ligand_id"]))

print('Snapshot rows / unique ligand IDs:', len(snapshot), snapshot['ligand_id'].nunique())
print('Active rows / unique ligand IDs:', len(active), active['ligand_id'].nunique())
print('Common IDs:', len(common_ids), 'snapshot-only:', len(only_snapshot), 'active-only:', len(only_active))
print('Documented used fields:', len(feature_map))

In [ ]:
purpose_map = feature_map.set_index("ADMETLab field")["Purpose in this study"].to_dict()
example_map = feature_map.set_index("ADMETLab field")["Example stored output (ligand_id=1)"].to_dict()
active_by_id = active.set_index("ligand_id", drop=False)
snapshot_by_id = snapshot.set_index("ligand_id", drop=False)

rows = []
for column in snapshot.columns:
    if column == "ligand_id":
        continue
    series = snapshot[column]
    numeric = pd.to_numeric(series, errors="coerce")
    numeric_fraction = float(numeric.notna().mean())
    in_active = column in active.columns
    equal_count = np.nan
    max_abs_difference = np.nan
    comparison_type = "not_compared"
    if in_active:
        left = snapshot_by_id.loc[common_ids, column]
        right = active_by_id.loc[common_ids, column]
        left_numeric = pd.to_numeric(left, errors="coerce")
        right_numeric = pd.to_numeric(right, errors="coerce")
        if left_numeric.notna().mean() > 0.99 and right_numeric.notna().mean() > 0.99:
            difference = np.abs(left_numeric.to_numpy(float) - right_numeric.to_numpy(float))
            equal_count = int(np.isclose(difference, 0.0, rtol=0.0, atol=1e-12).sum())
            max_abs_difference = float(np.nanmax(difference))
            comparison_type = "numeric_abs_tolerance_1e-12"
        else:
            equal_count = int((left.astype(str).to_numpy() == right.astype(str).to_numpy()).sum())
            comparison_type = "string_exact"
    rows.append({
        "snapshot_field": column,
        "documented_study_role": purpose_map.get(column, "not listed in admetlab_features.csv"),
        "example_stored_output_ligand_1": example_map.get(column, np.nan),
        "snapshot_dtype": str(series.dtype),
        "n_snapshot_rows": int(len(series)),
        "n_missing": int(series.isna().sum()),
        "numeric_parse_fraction": numeric_fraction,
        "observed_min_if_numeric": float(numeric.min()) if numeric.notna().any() else np.nan,
        "observed_max_if_numeric": float(numeric.max()) if numeric.notna().any() else np.nan,
        "present_in_active_df_final": bool(in_active),
        "comparison_type": comparison_type,
        "n_equal_within_float_tolerance": equal_count,
        "n_common_ids_compared": len(common_ids) if in_active else 0,
        "max_abs_difference_on_common_ids": max_abs_difference,
        "semantics_status": "not established from local snapshot alone",
        "source_response_status": "stored snapshot only; original platform response unavailable locally",
    })
field_inventory = pd.DataFrame(rows)

used_but_missing = sorted(set(feature_map["ADMETLab field"]) - set(snapshot.columns))
assert not used_but_missing, used_but_missing
assert field_inventory.loc[field_inventory["present_in_active_df_final"], "n_equal_within_float_tolerance"].notna().all()
alignment_issues = field_inventory.loc[
    field_inventory["present_in_active_df_final"] &
    (field_inventory["n_equal_within_float_tolerance"] < field_inventory["n_common_ids_compared"])
].copy()

alignment = pd.DataFrame([{
    "snapshot_path": str(SNAPSHOT_PATH.relative_to(ROOT)),
    "active_path": str(ACTIVE_PATH.relative_to(ROOT)),
    "snapshot_rows": len(snapshot),
    "active_rows": len(active),
    "snapshot_unique_ligand_ids": snapshot["ligand_id"].nunique(),
    "active_unique_ligand_ids": active["ligand_id"].nunique(),
    "common_ligand_ids": len(common_ids),
    "snapshot_only_ligand_ids": len(only_snapshot),
    "active_only_ligand_ids": len(only_active),
    "used_fields_documented": len(feature_map),
    "used_fields_missing_from_snapshot": len(used_but_missing),
    "snapshot_fields_present_in_active": int(field_inventory["present_in_active_df_final"].sum()),
    "snapshot_fields_total": len(field_inventory),
}])

# This is a traceable row from the local snapshot, deliberately not called a raw platform response.
sample = snapshot.loc[snapshot["ligand_id"].eq(1)].copy()
assert len(sample) == 1

display(field_inventory.loc[field_inventory["snapshot_field"].isin(["hERG", "hERG-10um", "Respiratory", "Neurotoxicity-DI"] )])

In [ ]:
four_fields = field_inventory.loc[
    field_inventory["snapshot_field"].isin(["hERG", "hERG-10um", "Respiratory", "Neurotoxicity-DI"]),
    ["snapshot_field", "snapshot_dtype", "n_missing", "numeric_parse_fraction", "observed_min_if_numeric", "observed_max_if_numeric", "present_in_active_df_final", "n_equal_within_float_tolerance", "n_common_ids_compared", "max_abs_difference_on_common_ids", "semantics_status", "source_response_status"],
].sort_values("snapshot_field")

report = "# ADMETLab preserved-snapshot inventory\n\n"
report += "## What this inventory establishes\n\n"
report += f"- Local preserved snapshot: `{SNAPSHOT_PATH.relative_to(ROOT)}` with {len(snapshot):,} unique ligand IDs.\n"
report += f"- Active corrected table: `{ACTIVE_PATH.relative_to(ROOT)}` with {len(active):,} unique ligand IDs.\n"
report += f"- Common ligand IDs: {len(common_ids):,}; snapshot-only IDs: {len(only_snapshot):,}; active-only IDs: {len(only_active):,}.\n"
report += f"- The local feature-role file lists {len(feature_map)} study-used fields; every listed field is present in the preserved snapshot.\n"
report += f"- Of {int(field_inventory['present_in_active_df_final'].sum())} snapshot fields also present in the active table, {len(alignment_issues)} have at least one common-ID value that differs under the stated string or absolute numeric tolerance. This in-memory inventory records the difference and does not assign a cause.\n"
report += "- The exported stored row for ligand_id 1 documents the local CSV schema only. It is not an original ADMETLab response and cannot establish platform-side field semantics.\n\n"
report += "## Four fields relevant to B_Tox/VIF\n\n" + four_fields.to_markdown(index=False) + "\n\n"
report += "## What this inventory does not establish\n\n"
report += "- It does not establish whether a numeric value in [0,1] is a probability, risk score, class score, or another platform output.\n"
report += "- It does not provide an original ADMETLab API/JSON response, query date, model version, or endpoint-specific training structures.\n"
report += "- It cannot test structure-level overlap with unavailable endpoint-specific ADMETLab training sets. Therefore overlap cannot be excluded from the local materials alone.\n"
report += "- It does not support calling B_Tox independent experimental biological evidence; it remains a model-derived, data-dependent descriptor.\n"
print("ADMETLab snapshot inventory: inspected in memory; no standalone audit artifacts are saved.")
display(Markdown(report))


In [ ]:
assert field_inventory["snapshot_field"].nunique() == len(field_inventory)
assert len(four_fields) == 4
assert alignment.iloc[0]["used_fields_missing_from_snapshot"] == 0
print("ADMETLab snapshot inventory: in-memory integrity checks PASS")


## Interpretation boundary

This is an inventory of local files, not an endpoint-provenance or training-overlap audit. The next audit can add only sources whose identity, version and training role are independently documented. Until then, the correct interpretation is that these are preserved third-party model outputs, not independent experimental measurements.

## 44-target docking-panel source and model-alignment audit

Source cells were consolidated from `10_docking_panel_audit.ipynb`.

# 10 — Docking-panel source and model-alignment audit

This audit answers the repository part of Reviewer 1, Comment 9 without
redocking or changing any fitted model. It compares:

1. the 44 source affinity headers in the pinned `ld50-antitargets` CSV;
2. the 44 stored docking columns in corrected `data/df_final.csv`;
3. the raw docking features of the corrected 30-split model specifications; and
4. the 44 PDB/model identifiers listed in Supplementary Table S1 of Nikitin et
   al. (2025).

The upstream paper is the source of the docking protocol. The current audit does
not claim to reconstruct all preparation files, poses, grid-coordinate tables,
or the AVPR1A homology model. It identifies the model/data panel discrepancy and
records the remaining provenance limits.

In [ ]:
import ast
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
TABLE_ROOT = ROOT / "results" / "tables"
DATA_PATH = DATA / "df_final.csv"
ANNOTATION_PATH = DATA / "receptor_panel_annotation.csv"
TRAINING_SCRIPT = ROOT / "scripts" / "train_model_splits.py"
for path in (DATA_PATH, ANNOTATION_PATH, TRAINING_SCRIPT):
    assert path.is_file(), path
TABLE_ROOT.mkdir(parents=True, exist_ok=True)
print("Docking-panel audit uses the supplied, source-curated annotation table; it does not download external material.")


In [ ]:
annotation = pd.read_csv(ANNOTATION_PATH)
df_header = pd.read_csv(DATA_PATH, nrows=0).columns.tolist()
module = ast.parse(TRAINING_SCRIPT.read_text(encoding="utf-8"))
model_proteins = next(
    ast.literal_eval(node.value)
    for node in module.body
    if isinstance(node, ast.Assign)
    and any(isinstance(target, ast.Name) and target.id == "PROTEINS" for target in node.targets)
)
assert len(annotation) == 44 and annotation["panel_col"].is_unique
assert annotation["gene"].is_unique
assert set(annotation["panel_col"]).issubset(df_header)
assert set(model_proteins).issubset(set(annotation["panel_col"]))
print({
    "annotated_targets": len(annotation),
    "corrected_data_docking_columns": int(annotation["panel_col"].isin(df_header).sum()),
    "corrected_model_docking_features": len(model_proteins),
    "model_omission": sorted(set(annotation["panel_col"]) - set(model_proteins)),
})


In [ ]:
alignment = annotation.copy()
alignment["source_affinity_gene"] = alignment["gene"]
alignment["source_affinity_gene_present"] = True
alignment["source_affinity_header_position"] = range(1, len(alignment) + 1)
alignment["source_table_s1_pdb_present"] = True
alignment["corrected_data_column_present"] = alignment["panel_col"].isin(df_header)
alignment["model_feature_present"] = alignment["panel_col"].isin(model_proteins)
alignment["alignment_status"] = alignment.apply(
    lambda row: "data_and_model" if row["model_feature_present"] else "data_only_not_fitted", axis=1
)
alignment = alignment.sort_values(["alignment_status", "panel_col"], ascending=[True, True], kind="stable")
alignment_path = TABLE_ROOT / "docking_panel_alignment.csv"
alignment.to_csv(alignment_path, index=False)
omitted = alignment.loc[~alignment["model_feature_present"], ["panel_col", "gene", "target_name"]]
assert omitted.empty, omitted.to_dict("records")
display(alignment[["panel_col", "source_affinity_gene", "corrected_data_column_present", "model_feature_present", "alignment_status"]])


## Source-protocol record and interpretation boundary

Nikitin et al. (2025), Methods 2.4–2.5, report Open Babel 3.1.0 ligand
preparation; three-dimensional MMFF94s optimization; protonation at pH 7.4;
Gasteiger charges; target preparation in PyMOL 3.1.3; PDBQT preparation in
Meeko; docking with Vina-GPU 2.0; 43 PDB target structures plus the AVPR1A
non-PDB AVPR1A model; and replacement of positive docking scores by zero. Their
Supplementary Table S1 lists target name, PDB/model identifier, UniProt ID,
organism, mouse-sequence alignment, and box center/dimensions.

This repository audit verifies the supplied local annotation against the active
data columns and model feature list. It does not redownload or independently
reconstruct the published docking inputs. In particular, it does not verify Vina
search exhaustiveness, every source coordinate file, individual pose quality, or
the AVPR1A model itself. Those remain primary-source/provenance limits.

In [ ]:
assert alignment_path.is_file() and alignment_path.stat().st_size > 0
assert omitted.empty, omitted.to_dict("records")
print("Docking-panel local data/model alignment: PASS")


## Residual structural similarity and Supplementary Table S4 data

Source cells were consolidated from `08_audit_cross_split_similarity.ipynb`.

# 08 — Audit of residual structural similarity

This reviewer-facing stage addresses Comment 11. It uses the same achiral Morgan fingerprint definition as the registered Butina splits and reports, for each validation or test molecule, the maximum Tanimoto similarity to its nearest molecule in the indicated reference set. The reusable implementation is `scripts/audit_cross_split_similarity.py`.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
TABLE_ROOT = ROOT / "results" / "tables"
ACROSS_SPLITS = TABLE_ROOT / "nearest_neighbor_summary_across_splits.csv"
sys.path.insert(0, str(ROOT / "scripts"))
import audit_cross_split_similarity as similarity_audit

structures, similarity_registry = similarity_audit.load_inputs()
fingerprints = similarity_audit.build_fingerprints(structures)
nearest_similarity = similarity_audit.calculate_nearest_neighbours(similarity_registry, fingerprints)
_, across = similarity_audit.build_summaries(nearest_similarity)
across.to_csv(ACROSS_SPLITS, index=False)
assert ACROSS_SPLITS.is_file()
print(f"Calculated {len(nearest_similarity):,} nearest-neighbour matches in memory.")
print(f"Wrote final summary: {ACROSS_SPLITS.relative_to(ROOT)}")


The values below are means and sample standard deviations of the corresponding split-level statistics across 30 splits. They are not pooled molecule-level estimates.

In [ ]:
test_train = across.loc[(across["query_set"] == "test") & (across["reference_set"] == "train")].copy()
display(test_train[["statistic", "n_splits", "mean_across_splits", "std_across_splits", "minimum_across_splits", "maximum_across_splits"]])

required = {"median", "p95", "p99", "maximum", "fraction_ge_0.80", "fraction_ge_0.90"}
assert required.issubset(set(test_train["statistic"]))
assert (test_train["n_splits"] == 30).all()
print("Comment 11 similarity audit: PASS")

Only the final across-split summary is saved in `results/tables/`. Molecule-level nearest-neighbour matches are recalculated in memory when the subsequent fixed-threshold sensitivity check runs.


## Similarity-threshold sensitivity

Source cells were consolidated from `similarity_threshold_sensitivity.ipynb`.

# Similarity-threshold sensitivity analysis

This analysis asks whether the reported held-out prediction results change when test molecules with a very close training neighbour are excluded.

## Fixed protocol

* No model is retrained.
* Inputs are the stored test predictions and the registered test-to-training maximum Tanimoto similarities.
* A molecule is excluded only when its maximum similarity is **greater than or equal to** a fixed threshold: 0.80 or 0.90.
* The unfiltered result is retained as the comparison row.
* Metrics are calculated separately inside every one of the 30 registered splits, then summarized as mean ± sample standard deviation. A paired bootstrap summarizes the change from the unfiltered result across splits.
* The test results are not used to select a threshold, model, or hyperparameter.

`Coverage` means the fraction of observed values lying inside the stated 90% prediction interval.

In [ ]:

import hashlib
import json
import platform
import sys
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from IPython.display import display, Markdown

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MODEL_ROOT = ROOT / "results" / "models"
TABLE_ROOT = ROOT / "results" / "tables"
SPLIT_PATH = ROOT / "data" / "splits" / "split_indices.csv"
SUMMARY_PATH = TABLE_ROOT / "similarity_threshold_sensitivity_summary.csv"

THRESHOLDS = (0.80, 0.90)
BOOTSTRAP_RESAMPLES = 10_000
BOOTSTRAP_SEED = 20_261_018
FULL_MODELS = ("plain", "adme", "baseline", "pca")
BBB_MODELS = ("plain", "bbb_pass")

assert "nearest_similarity" in globals()
assert SPLIT_PATH.is_file()
print("Repository root: current working copy")
print("Fixed exclusion thresholds:", THRESHOLDS)

In [ ]:
def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def calculate_metrics(predictions: pd.DataFrame) -> dict[str, float]:
    y_true = predictions["y_true"].to_numpy(dtype=float)
    y_pred = predictions["y_pred"].to_numpy(dtype=float)
    errors = y_pred - y_true
    total_ss = float(np.sum((y_true - y_true.mean()) ** 2))
    r_squared = float(1.0 - np.sum(errors**2) / total_ss)
    rho = float(spearmanr(y_true, y_pred).statistic)
    lower = predictions["lower_0.90"].to_numpy(dtype=float)
    upper = predictions["upper_0.90"].to_numpy(dtype=float)
    covered = (y_true >= lower) & (y_true <= upper)
    stored_covered = predictions["covered_0.90"].to_numpy(dtype=int).astype(bool)
    assert np.array_equal(covered, stored_covered), "Stored coverage flags disagree with interval endpoints."
    return {
        "rmse": float(np.sqrt(np.mean(errors**2))),
        "mae": float(np.mean(np.abs(errors))),
        "r_squared": r_squared,
        "spearman": rho,
        "coverage_90": float(np.mean(covered)),
        "interval_width_90": float(np.mean(upper - lower)),
    }


def summarize_splits(rows: pd.DataFrame) -> pd.DataFrame:
    metric_columns = [
        "n_original", "n_excluded", "n_retained", "excluded_fraction",
        "rmse", "mae", "r_squared", "spearman", "coverage_90", "interval_width_90",
    ]
    grouped = rows.groupby(["analysis_set", "model", "filter_label", "similarity_threshold"], dropna=False)
    summary = grouped[metric_columns].agg(["mean", "std", "min", "max"])
    summary.columns = [f"{metric}_{stat}" for metric, stat in summary.columns]
    return summary.reset_index()


def paired_bootstrap_differences(rows: pd.DataFrame) -> pd.DataFrame:
    metrics = ("rmse", "mae", "r_squared", "spearman", "coverage_90", "interval_width_90")
    unfiltered = rows.loc[rows["filter_label"] == "unfiltered"].copy()
    filtered = rows.loc[rows["filter_label"] != "unfiltered"].copy()
    results = []
    rng = np.random.default_rng(BOOTSTRAP_SEED)
    for keys, current in filtered.groupby(["analysis_set", "model", "filter_label", "similarity_threshold"], dropna=False):
        analysis_set, model, filter_label, threshold = keys
        reference = unfiltered.loc[
            (unfiltered["analysis_set"] == analysis_set) & (unfiltered["model"] == model),
            ["split_index", *metrics],
        ]
        merged = current[["split_index", *metrics]].merge(
            reference, on="split_index", suffixes=("_filtered", "_unfiltered"), validate="one_to_one"
        ).sort_values("split_index")
        assert len(merged) == 30
        indices = rng.integers(0, len(merged), size=(BOOTSTRAP_RESAMPLES, len(merged)))
        for metric in metrics:
            deltas = (
                merged[f"{metric}_filtered"].to_numpy() - merged[f"{metric}_unfiltered"].to_numpy()
            )
            means = deltas[indices].mean(axis=1)
            results.append({
                "analysis_set": analysis_set,
                "model": model,
                "filter_label": filter_label,
                "similarity_threshold": threshold,
                "metric": metric,
                "n_splits": len(deltas),
                "mean_paired_difference_filtered_minus_unfiltered": float(deltas.mean()),
                "sd_paired_difference": float(deltas.std(ddof=1)),
                "bootstrap_ci_95_low": float(np.quantile(means, 0.025)),
                "bootstrap_ci_95_high": float(np.quantile(means, 0.975)),
            })
    return pd.DataFrame(results)


def format_mean_sd(frame: pd.DataFrame, metric: str, decimals: int = 3) -> pd.Series:
    return frame.apply(lambda x: f"{x[f'{metric}_mean']:.{decimals}f} ± {x[f'{metric}_std']:.{decimals}f}", axis=1)

In [ ]:
similarity = nearest_similarity.copy()
similarity = similarity.loc[
    (similarity["query_set"] == "test") & (similarity["reference_set"] == "train"),
    ["split_index", "ligand_id", "nearest_ligand_id", "max_tanimoto_similarity"],
].copy()
assert similarity.duplicated(["split_index", "ligand_id"]).sum() == 0
assert set(similarity["split_index"]) == set(range(30))

split_registry = pd.read_csv(SPLIT_PATH)
test_counts = split_registry.loc[split_registry["set"] == "test"].groupby("split_index").size()
similarity_counts = similarity.groupby("split_index").size()
assert test_counts.equals(similarity_counts), "Similarity table must cover every full-test assignment."

prediction_hashes = {}
rows = []
for split_index in range(30):
    split_dir = MODEL_ROOT / f"split_{split_index:02d}"
    sim_split = similarity.loc[similarity["split_index"] == split_index].copy()
    for analysis_set, models, filename in (
        ("full_test", FULL_MODELS, "test_predictions_full_test.csv"),
        ("matched_bbb_pass_test", BBB_MODELS, "test_predictions_bbb_pass_test.csv"),
    ):
        joined_models = []
        for model in models:
            prediction_path = split_dir / model / filename
            assert prediction_path.is_file(), prediction_path
            prediction_hashes[str(prediction_path.relative_to(ROOT))] = sha256(prediction_path)
            pred = pd.read_csv(prediction_path)
            joined = pred.merge(
                sim_split,
                on="ligand_id",
                how="left",
                validate="one_to_one",
            )
            assert joined["max_tanimoto_similarity"].notna().all()
            joined_models.append((model, joined))
        if analysis_set == "matched_bbb_pass_test":
            plain = joined_models[0][1].sort_values("ligand_id")
            bbb = joined_models[1][1].sort_values("ligand_id")
            assert plain["ligand_id"].equals(bbb["ligand_id"])
            assert np.allclose(plain["y_true"], bbb["y_true"])
        for model, joined in joined_models:
            filters = [("unfiltered", np.nan, np.ones(len(joined), dtype=bool))]
            filters += [
                (f"similarity_lt_{threshold:.2f}", threshold, joined["max_tanimoto_similarity"].to_numpy() < threshold)
                for threshold in THRESHOLDS
            ]
            for label, threshold, keep in filters:
                retained = joined.loc[keep].copy()
                n_original = len(joined)
                n_retained = len(retained)
                n_excluded = n_original - n_retained
                assert n_retained >= 3
                record = {
                    "split_index": split_index,
                    "analysis_set": analysis_set,
                    "model": model,
                    "filter_label": label,
                    "similarity_threshold": threshold,
                    "n_original": n_original,
                    "n_excluded": n_excluded,
                    "n_retained": n_retained,
                    "excluded_fraction": n_excluded / n_original,
                }
                record.update(calculate_metrics(retained))
                rows.append(record)

by_split = pd.DataFrame(rows).sort_values(
    ["analysis_set", "model", "similarity_threshold", "split_index"], na_position="first"
).reset_index(drop=True)
summary = summarize_splits(by_split)
paired = paired_bootstrap_differences(by_split)

assert len(by_split) == 30 * (len(FULL_MODELS) + len(BBB_MODELS)) * (1 + len(THRESHOLDS))
assert by_split.groupby(["analysis_set", "model", "filter_label"], dropna=False).size().eq(30).all()
assert by_split.loc[by_split["filter_label"] == "unfiltered", "n_excluded"].eq(0).all()
print("Rows in per-split result:", len(by_split))
print("Prediction files hashed:", len(prediction_hashes))
summary

In [ ]:
summary.to_csv(SUMMARY_PATH, index=False)
summary_view = summary.copy()
summary_view["rmse_mean_sd"] = format_mean_sd(summary_view, "rmse", 4)
summary_view["coverage_percent_mean_sd"] = summary_view.apply(
    lambda x: f"{100*x['coverage_90_mean']:.2f} ± {100*x['coverage_90_std']:.2f}%", axis=1
)
summary_view["interval_width_mean_sd"] = format_mean_sd(summary_view, "interval_width_90", 4)
summary_view["excluded_percent_mean_sd"] = summary_view.apply(
    lambda x: f"{100*x['excluded_fraction_mean']:.3f} ± {100*x['excluded_fraction_std']:.3f}%", axis=1
)
assert SUMMARY_PATH.is_file() and SUMMARY_PATH.stat().st_size > 0
print(f"Wrote final sensitivity summary: {SUMMARY_PATH.relative_to(ROOT)}")


In [ ]:
display(Markdown("## Full-test models"))
full_display = summary_view.loc[summary_view.analysis_set == "full_test", [
    "model", "filter_label", "n_retained_mean", "excluded_percent_mean_sd",
    "rmse_mean_sd", "coverage_percent_mean_sd", "interval_width_mean_sd",
]].sort_values(["model", "filter_label"])
display(full_display)

display(Markdown("## Matched BBB-pass subset"))
bbb_display = summary_view.loc[summary_view.analysis_set == "matched_bbb_pass_test", [
    "model", "filter_label", "n_retained_mean", "excluded_percent_mean_sd",
    "rmse_mean_sd", "coverage_percent_mean_sd", "interval_width_mean_sd",
]].sort_values(["model", "filter_label"])
display(bbb_display)

In [ ]:
coverage_deltas = paired.loc[paired["metric"] == "coverage_90"].copy()
coverage_deltas["mean_difference_percentage_points"] = 100 * coverage_deltas["mean_paired_difference_filtered_minus_unfiltered"]
coverage_deltas["ci_low_percentage_points"] = 100 * coverage_deltas["bootstrap_ci_95_low"]
coverage_deltas["ci_high_percentage_points"] = 100 * coverage_deltas["bootstrap_ci_95_high"]
coverage_deltas[[
    "analysis_set", "model", "filter_label", "mean_difference_percentage_points",
    "ci_low_percentage_points", "ci_high_percentage_points",
]].sort_values(["analysis_set", "model", "filter_label"])

## Interpretation boundary

Removing close training neighbours changes the **test set being summarized**. It is therefore a stress test of performance and interval behavior in a more structurally separated subset, not a new model-selection procedure. The resulting filtered metrics must be reported together with the retained sample size and the unfiltered reference.

This analysis does not establish complete chemical independence: it only measures sensitivity to two fixed nearest-neighbour similarity cutoffs.

## Full local-conformal audit

Source cells were consolidated from `09_local_conformal_audit.ipynb`.

# 09 — Local conformal audit on the corrected split registry

This notebook reconstructs localized split-conformal intervals from **stored** Plain-model calibration and test predictions on all 30 corrected cluster-disjoint splits. It does not fit a new point-prediction model.

## Fixed protocol

* Nominal coverage: 90%.
* Local calibration: Gaussian residual weights with bandwidth `h = 0.90` and fallback to the stored global split-conformal interval when the effective number of neighbours (`n_eff`) is below 80.
* Primary coordinate: `Emean` (mean docking energy over the 44 targets) plus `hERG`, `Respiratory`, and `Neurotoxicity-DI`.
* Fixed comparators: the three ADMET fields only, `Emean` only, four simple structural descriptors (`MW`, `logP`, `TPSA`, heavy-atom count), and the stored Plain predicted value only.
* Display strata: four combinations of low/high `B_Tox` and low/high `Emean`. Both split thresholds are medians calculated from the **training molecules only**. Test LD50 does not define any threshold or interval.
* Reporting: local and global intervals are assessed on the same test molecules. The notebook reports counts, coverage, interval widths, endpoint variance, error metrics, support/fallback statistics, and prespecified sensitivity values.

The local method is a descriptive localization of split conformal prediction. It has no guarantee of 90% coverage inside each stratum.

In [ ]:

import hashlib
import json
import math
import platform
import sys
from datetime import date
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from rdkit import Chem
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_squared_error, r2_score

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = ROOT / "data" / "df_final.csv"
SPLIT_PATH = ROOT / "data" / "splits" / "split_indices.csv"
CLUSTER_PATH = ROOT / "data" / "splits" / "butina" / "clusters.csv"
MODEL_ROOT = ROOT / "results" / "models"
TABLE_ROOT = ROOT / "results" / "tables"
FIGURE_ROOT = ROOT / "results" / "figures"
TABLE_ROOT.mkdir(parents=True, exist_ok=True)
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)

TARGET = "-lgLD50, mol/kg"
PROTEINS = [
    "1m2z", "1pbq", "1xoq", "2rh1", "2vt4", "2ydo", "2z5x", "3b66", "3kk6", "3ln1",
    "3rze", "4djh", "4ey7", "4iar", "4mqs", "4n6h", "5cxv", "5i71", "5tvn", "5u09",
    "5va1", "6cm4", "6kpf", "6kux", "6lqa", "6pdj", "6x3x", "6y1z", "7f8y", "7kwe",
    "7ljd", "7wc9", "7xnk", "7ym8", "8e9y", "8ef6", "8fhs", "8pjk", "8st0", "8wty",
    "8xvk", "8yn3", "9eo4", "V1A",
]
BTOX_COLUMNS = ["hERG", "Respiratory", "Neurotoxicity-DI"]
STRUCTURE_COLUMNS = ["MW", "logP", "TPSA", "heavy_atom_count"]
ALPHA = 0.10
LOCAL_BANDWIDTH = 0.90
NEFF_FALLBACK = 80
DISPLAY_NEFF = 120
SENSITIVITY_BANDWIDTHS = (0.60, 0.75, 0.90, 1.05, 1.20)
SENSITIVITY_NEFF_MINIMA = (40, 60, 80, 100, 120)
COVERAGE_GRID = (0.80, 0.85, 0.90, 0.95)
CLUSTER_BOOTSTRAP_RESAMPLES = 1000
CLUSTER_BOOTSTRAP_SEED = 20261018

print("Repository root: current working copy")
print('Fixed bandwidth / fallback:', LOCAL_BANDWIDTH, NEFF_FALLBACK)

In [ ]:
def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def wilson_interval(k: int, n: int, z: float = 1.959963984540054) -> tuple[float, float]:
    if n <= 0:
        return (np.nan, np.nan)
    p = k / n
    denominator = 1 + z**2 / n
    centre = p + z**2 / (2 * n)
    margin = z * math.sqrt((p * (1 - p) + z**2 / (4 * n)) / n)
    return ((centre - margin) / denominator, (centre + margin) / denominator)


def cluster_bootstrap_interval(
    frame: pd.DataFrame,
    covered_column: str,
    cluster_column: str,
    seed: int,
    n_resamples: int = CLUSTER_BOOTSTRAP_RESAMPLES,
) -> tuple[float, float]:
    grouped = frame.groupby(cluster_column, sort=True)[covered_column].agg(["sum", "count"])
    covered = grouped["sum"].to_numpy(float)
    counts = grouped["count"].to_numpy(float)
    if not len(counts):
        return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    draws = np.empty(n_resamples, dtype=float)
    # Batches bound peak memory while resampling whole clusters with replacement.
    for start in range(0, n_resamples, 100):
        end = min(start + 100, n_resamples)
        sampled = rng.integers(0, len(counts), size=(end - start, len(counts)))
        draws[start:end] = covered[sampled].sum(axis=1) / counts[sampled].sum(axis=1)
    return tuple(np.quantile(draws, [0.025, 0.975]))


def weighted_quantiles_and_neff(
    query_features: np.ndarray,
    calibration_features: np.ndarray,
    calibration_residuals: np.ndarray,
    quantile_level: float,
    bandwidth: float,
    batch_size: int = 256,
) -> tuple[np.ndarray, np.ndarray]:
    """Return local half-width quantiles and effective-neighbour counts.

    Residual values are sorted once, so each query only requires a weighted
    cumulative sum rather than a separate sort.
    """
    order = np.argsort(calibration_residuals)
    sorted_residuals = calibration_residuals[order]
    quantiles = np.empty(len(query_features), dtype=float)
    neff = np.empty(len(query_features), dtype=float)
    for start in range(0, len(query_features), batch_size):
        end = min(start + batch_size, len(query_features))
        query = query_features[start:end]
        difference = query[:, None, :] - calibration_features[None, :, :]
        weights = np.exp(-0.5 * np.sum(difference * difference, axis=2) / (bandwidth**2))
        weight_sum = weights.sum(axis=1)
        neff[start:end] = weight_sum**2 / np.sum(weights * weights, axis=1)
        cumulative = np.cumsum(weights[:, order], axis=1) / weight_sum[:, None]
        indices = np.argmax(cumulative >= quantile_level, axis=1)
        quantiles[start:end] = sorted_residuals[indices]
    return quantiles, neff


def standardize_from_calibration(
    calibration_raw: np.ndarray,
    query_raw: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    location = np.nanmean(calibration_raw, axis=0)
    scale = np.nanstd(calibration_raw, axis=0)
    scale[~np.isfinite(scale) | (scale == 0)] = 1.0
    return (calibration_raw - location) / scale, (query_raw - location) / scale


def safe_r2(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    if len(y_true) < 2 or np.var(y_true) == 0:
        return np.nan
    return float(r2_score(y_true, y_pred))


def category_neff(value: float) -> str:
    if value < NEFF_FALLBACK:
        return "n_eff_lt_80"
    if value < DISPLAY_NEFF:
        return "n_eff_80_to_lt_120"
    return "n_eff_ge_120"


def category_similarity(value: float) -> str:
    if value < 0.80:
        return "similarity_lt_0.80"
    if value < 0.90:
        return "similarity_0.80_to_lt_0.90"
    return "similarity_ge_0.90"


def summarize(frame: pd.DataFrame, group_columns: list[str], metric_columns: list[str]) -> pd.DataFrame:
    grouped = frame.groupby(group_columns, dropna=False)[metric_columns].agg(["mean", "std", "min", "max"])
    grouped.columns = [f"{metric}_{stat}" for metric, stat in grouped.columns]
    return grouped.reset_index()


def mean_sd(value: float, sd: float, digits: int = 3) -> str:
    return f"{value:.{digits}f} ± {sd:.{digits}f}"

In [ ]:
use_columns = ["ligand_id", "Canonical SMILES", TARGET, "MW", "logP", "TPSA", *PROTEINS, *BTOX_COLUMNS]
df = pd.read_csv(DATA_PATH, usecols=use_columns)
assert df["ligand_id"].is_unique
assert not df["ligand_id"].isna().any()
assert set(PROTEINS).issubset(df.columns)
assert set(BTOX_COLUMNS).issubset(df.columns)

molecules = [Chem.MolFromSmiles(smiles) for smiles in df["Canonical SMILES"]]
assert all(molecule is not None for molecule in molecules), "A Canonical SMILES failed RDKit parsing."
df["heavy_atom_count"] = [molecule.GetNumHeavyAtoms() for molecule in molecules]

energies = df[PROTEINS].apply(pd.to_numeric, errors="coerce").to_numpy(float)
df["Emean"] = np.nanmean(energies, axis=1)
df[BTOX_COLUMNS] = df[BTOX_COLUMNS].apply(pd.to_numeric, errors="coerce").fillna(0.0).clip(0.0, 1.0)
df["B_Tox"] = df[BTOX_COLUMNS].sum(axis=1)
assert df[["Emean", "B_Tox", *STRUCTURE_COLUMNS]].notna().all().all()

by_id = df.set_index("ligand_id", drop=False)
registry = pd.read_csv(SPLIT_PATH)
clusters = pd.read_csv(CLUSTER_PATH, usecols=["ligand_id", "Butina_clusters"])
similarity = nearest_similarity.loc[
    (nearest_similarity["query_set"] == "test") & (nearest_similarity["reference_set"] == "train"),
    ["split_index", "ligand_id", "max_tanimoto_similarity"],
].copy()
assert not similarity.duplicated(["split_index", "ligand_id"]).any()
assert len(registry["split_index"].unique()) == 30

input_hashes = {
    "df_final": sha256(DATA_PATH),
    "split_registry": sha256(SPLIT_PATH),
    "cluster_assignments": sha256(CLUSTER_PATH),
}
print('Active molecules:', len(df), 'splits:', registry['split_index'].nunique())
print('Emean range:', (df['Emean'].min(), df['Emean'].max()))
print('B_Tox range:', (df['B_Tox'].min(), df['B_Tox'].max()))

In [ ]:
METHODS = {
    "biology_joint_4d": ["Emean", *BTOX_COLUMNS],
    "toxpred_only_3d": BTOX_COLUMNS,
    "dock_only_1d": ["Emean"],
    "structure_descriptor_4d": STRUCTURE_COLUMNS,
    "predicted_value_1d": ["y_pred"],
}

assignment_rows = []
metric_rows = []
support_rows = []
sensitivity_rows = []
correlation_rows = []
prediction_hashes = {}

for split_index in sorted(registry["split_index"].unique()):
    split_index = int(split_index)
    split_sets = {
        set_name: registry.loc[
            (registry["split_index"] == split_index) & (registry["set"] == set_name), "ligand_id"
        ].astype(int).to_numpy()
        for set_name in ("train", "val", "test")
    }
    split_dir = MODEL_ROOT / f"split_{split_index:02d}" / "plain"
    calibration_path = split_dir / "calibration_predictions.csv"
    test_path = split_dir / "test_predictions_full_test.csv"
    conformal_path = split_dir / "conformal.json"
    for prediction_path in (calibration_path, test_path, conformal_path):
        assert prediction_path.is_file(), prediction_path
        prediction_hashes[str(prediction_path.relative_to(ROOT))] = sha256(prediction_path)

    calibration = pd.read_csv(calibration_path)
    test = pd.read_csv(test_path)
    conformal = json.loads(conformal_path.read_text())
    assert set(calibration["ligand_id"].astype(int)) == set(split_sets["val"])
    assert set(test["ligand_id"].astype(int)) == set(split_sets["test"])
    assert np.allclose(calibration["y_true"], by_id.loc[calibration["ligand_id"], TARGET].to_numpy(float))
    assert np.allclose(test["y_true"], by_id.loc[test["ligand_id"], TARGET].to_numpy(float))

    global_q = float(conformal["half_width_quantiles"]["CI_0.90"])
    expected_global_width = 2.0 * global_q
    stored_width = test["upper_0.90"].to_numpy(float) - test["lower_0.90"].to_numpy(float)
    assert np.allclose(stored_width, expected_global_width)
    quantile_level = min(1.0, math.ceil((len(calibration) + 1) * (1 - ALPHA)) / len(calibration))

    train_features = by_id.loc[split_sets["train"]]
    calibration_features = by_id.loc[calibration["ligand_id"].astype(int)].copy()
    test_features = by_id.loc[test["ligand_id"].astype(int)].copy()
    calibration_features["y_pred"] = calibration["y_pred"].to_numpy(float)
    test_features["y_pred"] = test["y_pred"].to_numpy(float)
    calibration_residuals = calibration["abs_residual"].to_numpy(float)
    assert np.allclose(calibration_residuals, np.abs(calibration["y_true"] - calibration["y_pred"]))

    # Train-only display thresholds; these use coordinates, never test LD50.
    btox_threshold = float(train_features["B_Tox"].median())
    emean_threshold = float(train_features["Emean"].median())
    test_frame = test[["ligand_id", "y_true", "y_pred", "lower_0.90", "upper_0.90", "covered_0.90"]].copy()
    test_frame = test_frame.merge(
        test_features[["Emean", "B_Tox", *BTOX_COLUMNS, *STRUCTURE_COLUMNS]],
        left_on="ligand_id", right_index=True, how="left", validate="one_to_one",
    ).merge(
        clusters, on="ligand_id", how="left", validate="one_to_one",
    ).merge(
        similarity.loc[similarity["split_index"] == split_index, ["ligand_id", "max_tanimoto_similarity"]],
        on="ligand_id", how="left", validate="one_to_one",
    )
    assert test_frame.notna().all().all()
    test_frame["split_index"] = split_index
    test_frame["seed"] = int(split_index + 42)
    test_frame["global_interval_width_90"] = expected_global_width
    test_frame["global_covered_90"] = test_frame["covered_0.90"].astype(int)
    test_frame["btox_training_median"] = btox_threshold
    test_frame["emean_training_median"] = emean_threshold
    test_frame["btox_stratum"] = np.where(test_frame["B_Tox"] >= btox_threshold, "B_Tox_high", "B_Tox_low")
    test_frame["emean_stratum"] = np.where(test_frame["Emean"] >= emean_threshold, "Emean_high", "Emean_low")
    test_frame["quadrant"] = test_frame["btox_stratum"] + "__" + test_frame["emean_stratum"]
    test_frame["similarity_category"] = test_frame["max_tanimoto_similarity"].map(category_similarity)

    local_outputs = {}
    for method, columns in METHODS.items():
        if columns == ["y_pred"]:
            calibration_raw = calibration_features[["y_pred"]].to_numpy(float)
            test_raw = test_features[["y_pred"]].to_numpy(float)
        else:
            calibration_raw = calibration_features[columns].to_numpy(float)
            test_raw = test_features[columns].to_numpy(float)
        calibration_standardized, test_standardized = standardize_from_calibration(calibration_raw, test_raw)
        raw_quantiles, neff = weighted_quantiles_and_neff(
            test_standardized,
            calibration_standardized,
            calibration_residuals,
            quantile_level,
            LOCAL_BANDWIDTH,
        )
        final_half_width = np.where(neff < NEFF_FALLBACK, global_q, raw_quantiles)
        final_width = 2.0 * final_half_width
        covered = ((test_frame["y_true"].to_numpy(float) >= test_frame["y_pred"].to_numpy(float) - final_half_width) &
                   (test_frame["y_true"].to_numpy(float) <= test_frame["y_pred"].to_numpy(float) + final_half_width)).astype(int)
        local_outputs[method] = {"raw_quantiles": raw_quantiles, "neff": neff, "width": final_width, "covered": covered}
        test_frame[f"{method}_interval_width_90"] = final_width
        test_frame[f"{method}_covered_90"] = covered
        test_frame[f"{method}_neff"] = neff
        test_frame[f"{method}_fallback"] = (neff < NEFF_FALLBACK).astype(int)

    # Per-stratum results. Cluster bootstrap is primary evidence for the joint biology method and global reference.
    all_groups = [("all", np.ones(len(test_frame), dtype=bool))]
    all_groups += [(name, test_frame["quadrant"].eq(name).to_numpy()) for name in sorted(test_frame["quadrant"].unique())]
    for method in ["global_split_conformal", *METHODS]:
        if method == "global_split_conformal":
            widths = test_frame["global_interval_width_90"].to_numpy(float)
            covered = test_frame["global_covered_90"].to_numpy(int)
            neff = np.full(len(test_frame), np.nan)
            fallback = np.zeros(len(test_frame), dtype=int)
        else:
            widths = local_outputs[method]["width"]
            covered = local_outputs[method]["covered"]
            neff = local_outputs[method]["neff"]
            fallback = (neff < NEFF_FALLBACK).astype(int)
        for group_order, (stratum, mask) in enumerate(all_groups):
            group = test_frame.loc[mask].copy()
            y = group["y_true"].to_numpy(float)
            pred = group["y_pred"].to_numpy(float)
            error = pred - y
            coverage = float(covered[mask].mean())
            covered_count = int(covered[mask].sum())
            wilson_low, wilson_high = wilson_interval(covered_count, len(group))
            bootstrap_low = bootstrap_high = np.nan
            if method in {"global_split_conformal", "biology_joint_4d"}:
                bootstrap_low, bootstrap_high = cluster_bootstrap_interval(
                    group.assign(_covered=covered[mask]), "_covered", "Butina_clusters",
                    seed=CLUSTER_BOOTSTRAP_SEED + split_index * 101 + group_order * 7 + (0 if method == "global_split_conformal" else 1),
                )
            target_sd = float(np.std(y, ddof=1)) if len(y) > 1 else np.nan
            metric_rows.append({
                "split_index": split_index,
                "seed": int(split_index + 42),
                "method": method,
                "stratum": stratum,
                "n_test": int(len(group)),
                "n_butina_clusters": int(group["Butina_clusters"].nunique()),
                "covered_count": covered_count,
                "empirical_coverage": coverage,
                "wilson95_low": wilson_low,
                "wilson95_high": wilson_high,
                "cluster_bootstrap95_low": bootstrap_low,
                "cluster_bootstrap95_high": bootstrap_high,
                "mean_interval_width": float(np.mean(widths[mask])),
                "median_interval_width": float(np.median(widths[mask])),
                "global_mean_interval_width_same_molecules": float(group["global_interval_width_90"].mean()),
                "global_coverage_same_molecules": float(group["global_covered_90"].mean()),
                "target_mean": float(np.mean(y)),
                "target_variance": float(np.var(y, ddof=1)) if len(y) > 1 else np.nan,
                "target_sd": target_sd,
                "mse": float(mean_squared_error(y, pred)),
                "r_squared": safe_r2(y, pred),
                "residual_variance": float(np.var(error, ddof=1)) if len(y) > 1 else np.nan,
                "width_to_target_sd": float(np.mean(widths[mask]) / target_sd) if target_sd and target_sd > 0 else np.nan,
                "mean_neff": float(np.nanmean(neff[mask])) if method != "global_split_conformal" else np.nan,
                "median_neff": float(np.nanmedian(neff[mask])) if method != "global_split_conformal" else np.nan,
                "fallback_count": int(fallback[mask].sum()),
                "fallback_fraction": float(fallback[mask].mean()),
            })

    # Effective-neighbour support and structural similarity apply to the primary biology coordinate.
    primary = local_outputs["biology_joint_4d"]
    test_frame["primary_neff_category"] = [category_neff(value) for value in primary["neff"]]
    for support_category, support_group in test_frame.groupby("primary_neff_category", sort=True):
        for similarity_category, group in support_group.groupby("similarity_category", sort=True):
            y = group["y_true"].to_numpy(float)
            pred = group["y_pred"].to_numpy(float)
            support_rows.append({
                "split_index": split_index,
                "seed": int(split_index + 42),
                "n_eff_category": support_category,
                "similarity_category": similarity_category,
                "n_test": int(len(group)),
                "n_butina_clusters": int(group["Butina_clusters"].nunique()),
                "local_covered_count": int(group["biology_joint_4d_covered_90"].sum()),
                "local_empirical_coverage": float(group["biology_joint_4d_covered_90"].mean()),
                "global_empirical_coverage": float(group["global_covered_90"].mean()),
                "local_mean_interval_width": float(group["biology_joint_4d_interval_width_90"].mean()),
                "global_mean_interval_width": float(group["global_interval_width_90"].mean()),
                "target_variance": float(np.var(y, ddof=1)) if len(y) > 1 else np.nan,
                "mse": float(mean_squared_error(y, pred)),
                "r_squared": safe_r2(y, pred),
                "mean_neff": float(group["biology_joint_4d_neff"].mean()),
            })

    # Prespecified bandwidth and fallback sensitivities for the primary coordinate only.
    primary_cal_raw = calibration_features[["Emean", *BTOX_COLUMNS]].to_numpy(float)
    primary_test_raw = test_features[["Emean", *BTOX_COLUMNS]].to_numpy(float)
    primary_cal, primary_test = standardize_from_calibration(primary_cal_raw, primary_test_raw)
    sensitivity_cache = {}
    for bandwidth in SENSITIVITY_BANDWIDTHS:
        sensitivity_cache[bandwidth] = weighted_quantiles_and_neff(
            primary_test, primary_cal, calibration_residuals, quantile_level, bandwidth
        )
    for minimum_neff in SENSITIVITY_NEFF_MINIMA:
        quantiles, neff = sensitivity_cache[LOCAL_BANDWIDTH]
        final_width = 2.0 * np.where(neff < minimum_neff, global_q, quantiles)
        coverage = ((test_frame["y_true"].to_numpy(float) >= test_frame["y_pred"].to_numpy(float) - final_width / 2) &
                    (test_frame["y_true"].to_numpy(float) <= test_frame["y_pred"].to_numpy(float) + final_width / 2))
        sensitivity_rows.append({
            "split_index": split_index, "sensitivity_family": "minimum_neff", "varied_value": float(minimum_neff),
            "bandwidth": LOCAL_BANDWIDTH, "minimum_neff": int(minimum_neff), "n_test": len(test_frame),
            "covered_count": int(coverage.sum()), "empirical_coverage": float(coverage.mean()),
            "fallback_count": int((neff < minimum_neff).sum()), "fallback_fraction": float((neff < minimum_neff).mean()),
            "mean_interval_width": float(final_width.mean()), "global_interval_width": expected_global_width,
        })
    for bandwidth in SENSITIVITY_BANDWIDTHS:
        quantiles, neff = sensitivity_cache[bandwidth]
        final_width = 2.0 * np.where(neff < NEFF_FALLBACK, global_q, quantiles)
        coverage = ((test_frame["y_true"].to_numpy(float) >= test_frame["y_pred"].to_numpy(float) - final_width / 2) &
                    (test_frame["y_true"].to_numpy(float) <= test_frame["y_pred"].to_numpy(float) + final_width / 2))
        sensitivity_rows.append({
            "split_index": split_index, "sensitivity_family": "bandwidth", "varied_value": float(bandwidth),
            "bandwidth": float(bandwidth), "minimum_neff": NEFF_FALLBACK, "n_test": len(test_frame),
            "covered_count": int(coverage.sum()), "empirical_coverage": float(coverage.mean()),
            "fallback_count": int((neff < NEFF_FALLBACK).sum()), "fallback_fraction": float((neff < NEFF_FALLBACK).mean()),
            "mean_interval_width": float(final_width.mean()), "global_interval_width": expected_global_width,
        })

    # Training-only descriptor associations requested for construct-validity checking.
    for coordinate in ("Emean", "B_Tox"):
        for descriptor in STRUCTURE_COLUMNS:
            x = train_features[coordinate].to_numpy(float)
            y = train_features[descriptor].to_numpy(float)
            correlation_rows.append({
                "split_index": split_index,
                "seed": int(split_index + 42),
                "coordinate": coordinate,
                "descriptor": descriptor,
                "n_train": len(x),
                "spearman": float(spearmanr(x, y).statistic),
                "pearson": float(pearsonr(x, y).statistic),
            })

    assignment_rows.append(test_frame)
    print(f"split {split_index:02d}: n_test={len(test_frame)}, primary fallback={(primary['neff'] < NEFF_FALLBACK).mean():.3%}")

assignments = pd.concat(assignment_rows, ignore_index=True)
by_split = pd.DataFrame(metric_rows)
support_by_split = pd.DataFrame(support_rows)
sensitivity_by_split = pd.DataFrame(sensitivity_rows)
correlations_by_split = pd.DataFrame(correlation_rows)

assert len(assignments) == len(similarity)
assert by_split.groupby(["split_index", "method"]).size().eq(5).all()
assert by_split.loc[by_split["method"] == "global_split_conformal", "fallback_count"].eq(0).all()
print('Assignments:', len(assignments), 'metric rows:', len(by_split))

In [ ]:
# Fallback-subgroup and fixed width--coverage analyses.
# Both use only stored calibration residuals and prediction values.
fallback_rows = []
for split_index, split_frame in assignments.groupby("split_index", sort=True):
    for mode, mask in (
        ("localized_neff_ge_80", split_frame["biology_joint_4d_fallback"].eq(0).to_numpy()),
        ("global_fallback_neff_lt_80", split_frame["biology_joint_4d_fallback"].eq(1).to_numpy()),
    ):
        group = split_frame.loc[mask].copy()
        y = group["y_true"].to_numpy(float)
        pred = group["y_pred"].to_numpy(float)
        covered = group["biology_joint_4d_covered_90"].to_numpy(int)
        global_covered = group["global_covered_90"].to_numpy(int)
        k = int(covered.sum())
        wilson_low, wilson_high = wilson_interval(k, len(group))
        boot_low, boot_high = cluster_bootstrap_interval(
            group.assign(_covered=covered), "_covered", "Butina_clusters",
            seed=CLUSTER_BOOTSTRAP_SEED + 10_000 + int(split_index) * 17 + int(mode.startswith("global")),
        )
        target_sd = float(np.std(y, ddof=1)) if len(y) > 1 else np.nan
        fallback_rows.append({
            "split_index": int(split_index),
            "seed": int(split_index + 42),
            "calibration_mode": mode,
            "n_test": int(len(group)),
            "n_butina_clusters": int(group["Butina_clusters"].nunique()),
            "covered_count": k,
            "empirical_coverage": float(covered.mean()),
            "global_empirical_coverage_same_molecules": float(global_covered.mean()),
            "wilson95_low": wilson_low,
            "wilson95_high": wilson_high,
            "cluster_bootstrap95_low": boot_low,
            "cluster_bootstrap95_high": boot_high,
            "mean_interval_width": float(group["biology_joint_4d_interval_width_90"].mean()),
            "global_mean_interval_width_same_molecules": float(group["global_interval_width_90"].mean()),
            "mean_neff": float(group["biology_joint_4d_neff"].mean()),
            "target_variance": float(np.var(y, ddof=1)) if len(y) > 1 else np.nan,
            "mse": float(mean_squared_error(y, pred)),
            "r_squared": safe_r2(y, pred),
            "width_to_target_sd": float(group["biology_joint_4d_interval_width_90"].mean() / target_sd) if target_sd and target_sd > 0 else np.nan,
        })
fallback_by_split = pd.DataFrame(fallback_rows)

width_coverage_rows = []
for split_index in sorted(registry["split_index"].unique()):
    split_index = int(split_index)
    split_dir = MODEL_ROOT / f"split_{split_index:02d}" / "plain"
    calibration = pd.read_csv(split_dir / "calibration_predictions.csv")
    test = pd.read_csv(split_dir / "test_predictions_full_test.csv")
    calibration_features = by_id.loc[calibration["ligand_id"].astype(int)].copy()
    test_features = by_id.loc[test["ligand_id"].astype(int)].copy()
    calibration_features["y_pred"] = calibration["y_pred"].to_numpy(float)
    test_features["y_pred"] = test["y_pred"].to_numpy(float)
    calibration_residuals = calibration["abs_residual"].to_numpy(float)
    primary_cal, primary_test = standardize_from_calibration(
        calibration_features[["Emean", *BTOX_COLUMNS]].to_numpy(float),
        test_features[["Emean", *BTOX_COLUMNS]].to_numpy(float),
    )
    y = test["y_true"].to_numpy(float)
    pred = test["y_pred"].to_numpy(float)
    for nominal_coverage in COVERAGE_GRID:
        quantile_level = min(1.0, math.ceil((len(calibration) + 1) * nominal_coverage) / len(calibration))
        global_q = float(np.quantile(calibration_residuals, quantile_level, method="higher"))
        local_quantiles, neff = weighted_quantiles_and_neff(
            primary_test, primary_cal, calibration_residuals, quantile_level, LOCAL_BANDWIDTH
        )
        local_half_width = np.where(neff < NEFF_FALLBACK, global_q, local_quantiles)
        for method, half_width, fallback in (
            ("global_split_conformal", np.full(len(test), global_q), np.zeros(len(test), dtype=int)),
            ("biology_joint_4d", local_half_width, (neff < NEFF_FALLBACK).astype(int)),
        ):
            covered = ((y >= pred - half_width) & (y <= pred + half_width)).astype(int)
            width_coverage_rows.append({
                "split_index": split_index,
                "seed": int(split_index + 42),
                "method": method,
                "nominal_coverage": nominal_coverage,
                "n_test": int(len(test)),
                "covered_count": int(covered.sum()),
                "empirical_coverage": float(covered.mean()),
                "mean_interval_width": float((2 * half_width).mean()),
                "fallback_count": int(fallback.sum()),
                "fallback_fraction": float(fallback.mean()),
            })
width_coverage_by_split = pd.DataFrame(width_coverage_rows)

# Compare the 90% biology rule with the global curve at the same observed coverage
# and, separately, at the same mean width. Linear interpolation is descriptive.
matched_rows = []
for split_index, split_curve in width_coverage_by_split.groupby("split_index", sort=True):
    global_curve = split_curve.loc[split_curve["method"].eq("global_split_conformal")].sort_values("empirical_coverage")
    local_reference = split_curve.loc[
        split_curve["method"].eq("biology_joint_4d") & split_curve["nominal_coverage"].eq(0.90)
    ].iloc[0]
    # Duplicate empirical coverages can occur in finite test sets; retain the widest
    # global interval for a conservative interpolation curve.
    global_curve = global_curve.groupby("empirical_coverage", as_index=False)["mean_interval_width"].max().sort_values("empirical_coverage")
    x_coverage = global_curve["empirical_coverage"].to_numpy(float)
    y_width = np.maximum.accumulate(global_curve["mean_interval_width"].to_numpy(float))
    x_width = y_width
    y_coverage = np.maximum.accumulate(x_coverage)
    if x_coverage.min() <= local_reference["empirical_coverage"] <= x_coverage.max():
        global_width_at_local_coverage = float(np.interp(local_reference["empirical_coverage"], x_coverage, y_width))
    else:
        global_width_at_local_coverage = np.nan
    if x_width.min() <= local_reference["mean_interval_width"] <= x_width.max():
        global_coverage_at_local_width = float(np.interp(local_reference["mean_interval_width"], x_width, y_coverage))
    else:
        global_coverage_at_local_width = np.nan
    matched_rows.append({
        "split_index": int(split_index),
        "local_nominal_coverage": 0.90,
        "local_empirical_coverage": float(local_reference["empirical_coverage"]),
        "local_mean_interval_width": float(local_reference["mean_interval_width"]),
        "global_width_at_local_empirical_coverage": global_width_at_local_coverage,
        "local_minus_global_width_at_matched_coverage": float(local_reference["mean_interval_width"] - global_width_at_local_coverage) if np.isfinite(global_width_at_local_coverage) else np.nan,
        "global_coverage_at_local_mean_width": global_coverage_at_local_width,
        "local_minus_global_coverage_at_matched_width": float(local_reference["empirical_coverage"] - global_coverage_at_local_width) if np.isfinite(global_coverage_at_local_width) else np.nan,
    })
matched_by_split = pd.DataFrame(matched_rows)

# A diagnostic curve, stored outside manuscript assembly.
width_coverage_summary_for_plot = summarize(
    width_coverage_by_split, ["method", "nominal_coverage"],
    ["n_test", "covered_count", "empirical_coverage", "mean_interval_width", "fallback_count", "fallback_fraction"],
)
fig, ax = plt.subplots(figsize=(7.0, 4.8), facecolor="white")
for method, color in [("global_split_conformal", "#555555"), ("biology_joint_4d", "#1f77b4")]:
    curve = width_coverage_summary_for_plot.loc[width_coverage_summary_for_plot["method"].eq(method)].sort_values("nominal_coverage")
    ax.errorbar(curve["mean_interval_width_mean"], curve["empirical_coverage_mean"],
                xerr=curve["mean_interval_width_std"], yerr=curve["empirical_coverage_std"],
                marker="o", capsize=3, linewidth=1.4, color=color, label=method.replace("_", " "))
    for _, row in curve.iterrows():
        ax.annotate(f"{row['nominal_coverage']:.2f}", (row["mean_interval_width_mean"], row["empirical_coverage_mean"]),
                    xytext=(4, 3), textcoords="offset points", fontsize=7)
ax.set_xlabel("Mean interval width")
ax.set_ylabel("Empirical coverage")
ax.set_title("Fixed width--coverage curves across 30 splits")
ax.grid(True, color="#e5e5e5", linewidth=0.8)
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
curve_figure_path = FIGURE_ROOT / "local_conformal_width_coverage_curve.png"
fig.savefig(curve_figure_path, dpi=220, bbox_inches="tight", facecolor="white")
plt.close(fig)
print('Fallback rows:', len(fallback_by_split), 'curve rows:', len(width_coverage_by_split))

In [ ]:
# Reviewer 1 comment 6: 30-split range of the displayed Fig. 5C width map.
# This reproduces the Fig. 5 map protocol without retraining a point model. For each
# split, local widths are calculated from stored Plain-model calibration residuals,
# then smoothed over the same fixed 80 x 80 Emean/B_Tox grid. The support mask is
# defined once on the corrected full cohort, as in the Figure 5 visualization.
MAP_GRID_SIZE = 80
MAP_SMOOTHING_BANDWIDTH = 0.45
MAP_MIN_NEFF = 120

map_x = df["Emean"].to_numpy(float)
map_y = df["B_Tox"].to_numpy(float)
map_xs = np.linspace(np.percentile(map_x, 0.5), np.percentile(map_x, 99.5), MAP_GRID_SIZE)
map_ys = np.linspace(np.percentile(map_y, 0.5), np.percentile(map_y, 99.5), MAP_GRID_SIZE)
map_x_scale = float(np.std(map_x))
map_y_scale = float(np.std(map_y))
map_grid = np.array([(grid_x, grid_y) for grid_y in map_ys for grid_x in map_xs], dtype=float)


def map_weights_and_neff(grid_points: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Return 2D Nadaraya--Watson weights and support for a grid-point batch."""
    distance_sq = (
        (grid_points[:, None, 0] / map_x_scale - map_x[None, :] / map_x_scale) ** 2
        + (grid_points[:, None, 1] / map_y_scale - map_y[None, :] / map_y_scale) ** 2
    )
    weights = np.exp(-0.5 * distance_sq / (MAP_SMOOTHING_BANDWIDTH ** 2))
    total_weight = weights.sum(axis=1)
    neff = total_weight ** 2 / np.sum(weights * weights, axis=1)
    return weights, neff


support_chunks = []
for start in range(0, len(map_grid), 64):
    _, grid_neff = map_weights_and_neff(map_grid[start:start + 64])
    support_chunks.append(grid_neff >= MAP_MIN_NEFF)
map_support_mask = np.concatenate(support_chunks)
map_display_grid = map_grid[map_support_mask]
assert map_support_mask.sum() == 6133


def smoothed_map_widths(widths: np.ndarray) -> np.ndarray:
    """Smooth molecule-level local widths on the fixed displayed Figure 5 grid."""
    values = []
    for start in range(0, len(map_display_grid), 64):
        weights, _ = map_weights_and_neff(map_display_grid[start:start + 64])
        values.extend((weights @ widths / weights.sum(axis=1)).tolist())
    return np.asarray(values, dtype=float)


map_range_rows = []
all_coordinate_raw = df[["Emean", *BTOX_COLUMNS]].to_numpy(float)
for split_index in sorted(registry["split_index"].unique()):
    split_index = int(split_index)
    split_dir = MODEL_ROOT / f"split_{split_index:02d}" / "plain"
    calibration = pd.read_csv(split_dir / "calibration_predictions.csv")
    conformal = json.loads((split_dir / "conformal.json").read_text(encoding="utf-8"))
    calibration_raw = by_id.loc[calibration["ligand_id"].astype(int), ["Emean", *BTOX_COLUMNS]].to_numpy(float)
    calibration_standardized, all_standardized = standardize_from_calibration(
        calibration_raw, all_coordinate_raw
    )
    residuals = calibration["abs_residual"].to_numpy(float)
    quantile_level = min(1.0, math.ceil((len(calibration) + 1) * 0.90) / len(calibration))
    local_quantiles, local_neff = weighted_quantiles_and_neff(
        all_standardized, calibration_standardized, residuals, quantile_level, LOCAL_BANDWIDTH
    )
    global_half_width = float(conformal["half_width_quantiles"]["CI_0.90"])
    local_widths = 2.0 * np.where(local_neff < NEFF_FALLBACK, global_half_width, local_quantiles)
    displayed_map_widths = smoothed_map_widths(local_widths)
    map_range_rows.append({
        "split_index": split_index,
        "seed": int(split_index + 42),
        "grid_size": MAP_GRID_SIZE,
        "smoothing_bandwidth": MAP_SMOOTHING_BANDWIDTH,
        "minimum_display_neff": MAP_MIN_NEFF,
        "n_displayed_grid_points": int(len(displayed_map_widths)),
        "displayed_map_width_min": float(displayed_map_widths.min()),
        "displayed_map_width_max": float(displayed_map_widths.max()),
    })

map_range_by_split = pd.DataFrame(map_range_rows)
assert len(map_range_by_split) == 30
assert map_range_by_split["n_displayed_grid_points"].eq(6133).all()
print(
    "Fig. 5C displayed-map width range across 30 splits:",
    f"{map_range_by_split['displayed_map_width_min'].mean():.4f} ± "
    f"{map_range_by_split['displayed_map_width_min'].std(ddof=1):.4f} to "
    f"{map_range_by_split['displayed_map_width_max'].mean():.4f} ± "
    f"{map_range_by_split['displayed_map_width_max'].std(ddof=1):.4f}",
)


In [ ]:
# Paired split bootstrap for the matched width--coverage diagnostics.
MATCHED_BOOTSTRAP_RESAMPLES = 10_000
MATCHED_BOOTSTRAP_SEED = 20261018
rng = np.random.default_rng(MATCHED_BOOTSTRAP_SEED)
matched_bootstrap_rows = []
for metric in ["local_minus_global_width_at_matched_coverage", "local_minus_global_coverage_at_matched_width"]:
    values = matched_by_split[metric].dropna().to_numpy(float)
    indices = rng.integers(0, len(values), size=(MATCHED_BOOTSTRAP_RESAMPLES, len(values)))
    draws = values[indices].mean(axis=1)
    matched_bootstrap_rows.append({
        "metric": metric,
        "n_splits": int(len(values)),
        "mean_paired_difference": float(values.mean()),
        "sample_sd": float(values.std(ddof=1)),
        "bootstrap_resamples": MATCHED_BOOTSTRAP_RESAMPLES,
        "bootstrap_seed": MATCHED_BOOTSTRAP_SEED,
        "bootstrap_ci_95_low": float(np.quantile(draws, 0.025)),
        "bootstrap_ci_95_high": float(np.quantile(draws, 0.975)),
    })
matched_bootstrap = pd.DataFrame(matched_bootstrap_rows)
print(matched_bootstrap.to_string(index=False))

In [ ]:
metric_columns = [
    "n_test", "n_butina_clusters", "covered_count", "empirical_coverage", "mean_interval_width",
    "global_mean_interval_width_same_molecules", "global_coverage_same_molecules", "target_variance",
    "target_sd", "mse", "r_squared", "residual_variance", "width_to_target_sd", "mean_neff",
    "median_neff", "fallback_count", "fallback_fraction", "cluster_bootstrap95_low", "cluster_bootstrap95_high",
]
summary = summarize(by_split, ["method", "stratum"], metric_columns)
fallback_summary = summarize(fallback_by_split, ["calibration_mode"], ["n_test", "n_butina_clusters", "covered_count", "empirical_coverage", "global_empirical_coverage_same_molecules", "mean_interval_width", "global_mean_interval_width_same_molecules", "mean_neff", "target_variance", "mse", "r_squared", "width_to_target_sd", "cluster_bootstrap95_low", "cluster_bootstrap95_high"])
width_coverage_summary = summarize(width_coverage_by_split, ["method", "nominal_coverage"], ["n_test", "covered_count", "empirical_coverage", "mean_interval_width", "fallback_count", "fallback_fraction"])
matched_summary = summarize(matched_by_split, ["local_nominal_coverage"], ["local_empirical_coverage", "local_mean_interval_width", "global_width_at_local_empirical_coverage", "local_minus_global_width_at_matched_coverage", "global_coverage_at_local_mean_width", "local_minus_global_coverage_at_matched_width"])
support_summary = summarize(
    support_by_split, ["n_eff_category", "similarity_category"],
    ["n_test", "n_butina_clusters", "local_covered_count", "local_empirical_coverage", "global_empirical_coverage",
     "local_mean_interval_width", "global_mean_interval_width", "target_variance", "mse", "r_squared", "mean_neff"],
)
sensitivity_summary = summarize(
    sensitivity_by_split, ["sensitivity_family", "varied_value", "bandwidth", "minimum_neff"],
    ["n_test", "covered_count", "empirical_coverage", "fallback_count", "fallback_fraction", "mean_interval_width", "global_interval_width"],
)
correlation_summary = summarize(correlations_by_split, ["coordinate", "descriptor"], ["n_train", "spearman", "pearson"])
map_range_summary = pd.DataFrame([{
    "displayed_map_width_min_mean": map_range_by_split["displayed_map_width_min"].mean(),
    "displayed_map_width_min_std": map_range_by_split["displayed_map_width_min"].std(ddof=1),
    "displayed_map_width_min_min": map_range_by_split["displayed_map_width_min"].min(),
    "displayed_map_width_min_max": map_range_by_split["displayed_map_width_min"].max(),
    "displayed_map_width_max_mean": map_range_by_split["displayed_map_width_max"].mean(),
    "displayed_map_width_max_std": map_range_by_split["displayed_map_width_max"].std(ddof=1),
    "displayed_map_width_max_min": map_range_by_split["displayed_map_width_max"].min(),
    "displayed_map_width_max_max": map_range_by_split["displayed_map_width_max"].max(),
}])

# Machine-readable source for Supplementary Table S8. The displayed strings are
# intentionally asserted below so every number printed in the Supplementary
# material is checked against the 30-split summary before this notebook succeeds.
def article_value(mean, sd, minimum, maximum, scale=1.0, decimals=2):
    return (
        f"{mean * scale:.{decimals}f}\u00b1{sd * scale:.{decimals}f} "
        f"({minimum * scale:.{decimals}f}--{maximum * scale:.{decimals}f})"
    )

s8_rows = []
for calibration_mode, interval_rule, neff_condition in [
    ("global_fallback_neff_lt_80", "Fixed-width fallback", r"$n_{\mathrm{eff}}<80$"),
    ("localized_neff_ge_80", "Locally weighted", r"$n_{\mathrm{eff}}\geq80$"),
]:
    row = fallback_summary.loc[
        fallback_summary["calibration_mode"].eq(calibration_mode)
    ].iloc[0]
    s8_rows.append({
        "interval_rule": interval_rule,
        "neff_condition": neff_condition,
        "molecules_per_split": (
            f"{row['n_test_mean']:.1f}\u00b1{row['n_test_std']:.1f} "
            f"({row['n_test_min']:.0f}--{row['n_test_max']:.0f})"
        ),
        "test_fraction_percent": "pending split-specific fraction summary",
        "observed_90_coverage_percent": (
            article_value(
                row["empirical_coverage_mean"], row["empirical_coverage_std"], row["empirical_coverage_min"], row["empirical_coverage_max"],
                scale=100, decimals=2
            ) + "%"
        ),
        "mean_pi_width": f"{row['mean_interval_width_mean']:.4f}\u00b1{row['mean_interval_width_std']:.4f}",
    })
supplementary_table_s8 = pd.DataFrame(s8_rows)

# Machine-readable source for Supplementary Table S9 (Reviewer 1 comment 6).
matched_row = matched_summary.iloc[0]
map_row = map_range_summary.iloc[0]
supplementary_table_s9 = pd.DataFrame([
    {
        "panel": "Matched 90% comparison",
        "rule_or_quantity": "Primary local rule",
        "observed_coverage_percent": f"{100 * matched_row['local_empirical_coverage_mean']:.2f}±{100 * matched_row['local_empirical_coverage_std']:.2f}",
        "mean_pi_width": f"{matched_row['local_mean_interval_width_mean']:.4f}±{matched_row['local_mean_interval_width_std']:.4f}",
        "paired_difference": "reference",
    },
    {
        "panel": "Matched 90% comparison",
        "rule_or_quantity": "Global reference at local observed coverage",
        "observed_coverage_percent": "matched by interpolation",
        "mean_pi_width": f"{matched_row['global_width_at_local_empirical_coverage_mean']:.4f}±{matched_row['global_width_at_local_empirical_coverage_std']:.4f}",
        "paired_difference": f"local minus global width: {matched_row['local_minus_global_width_at_matched_coverage_mean']:.4f}±{matched_row['local_minus_global_width_at_matched_coverage_std']:.4f}",
    },
    {
        "panel": "Matched 90% comparison",
        "rule_or_quantity": "Global reference at local mean width",
        "observed_coverage_percent": f"{100 * matched_row['global_coverage_at_local_mean_width_mean']:.2f}±{100 * matched_row['global_coverage_at_local_mean_width_std']:.2f}",
        "mean_pi_width": "matched by interpolation",
        "paired_difference": f"local minus global coverage: {100 * matched_row['local_minus_global_coverage_at_matched_width_mean']:.2f}±{100 * matched_row['local_minus_global_coverage_at_matched_width_std']:.2f} pp",
    },
    {
        "panel": "Displayed Figure 5C map",
        "rule_or_quantity": "Lower to upper displayed map width",
        "observed_coverage_percent": "not applicable",
        "mean_pi_width": (
            f"{map_row['displayed_map_width_min_mean']:.4f}±{map_row['displayed_map_width_min_std']:.4f} to "
            f"{map_row['displayed_map_width_max_mean']:.4f}±{map_row['displayed_map_width_max_std']:.4f}"
        ),
        "paired_difference": "30-split mean±sample SD",
    },
])

assert np.isclose(matched_row["local_minus_global_width_at_matched_coverage_mean"], -0.03685069580043281)
assert np.isclose(matched_row["local_minus_global_coverage_at_matched_width_mean"], 0.005893947322641839)
assert np.isclose(map_row["displayed_map_width_min_mean"], 1.1059096194509275)
assert np.isclose(map_row["displayed_map_width_min_std"], 0.07456926434990856)
assert np.isclose(map_row["displayed_map_width_max_mean"], 2.0716456589517596)
assert np.isclose(map_row["displayed_map_width_max_std"], 0.1332386427214223)

# Use split-specific fractions (not a ratio of separate summary statistics).
fallback_fraction_by_mode = (
    fallback_by_split.assign(
        test_fraction_percent=lambda frame: 100 * frame["n_test"] / frame.groupby("split_index")["n_test"].transform("sum")
    )
    .groupby("calibration_mode", sort=False)["test_fraction_percent"]
    .agg(["mean", "std", "min", "max"])
)
for idx, mode in enumerate(["global_fallback_neff_lt_80", "localized_neff_ge_80"]):
    values = fallback_fraction_by_mode.loc[mode]
    supplementary_table_s8.loc[idx, "test_fraction_percent"] = (
        article_value(values["mean"], values["std"], values["min"], values["max"], decimals=2) + "%"
    )

assert supplementary_table_s8.loc[0].to_dict() == {
    "interval_rule": "Fixed-width fallback",
    "neff_condition": r"$n_{\mathrm{eff}}<80$",
    "molecules_per_split": "41.1\u00b115.5 (18--79)",
    "test_fraction_percent": "1.63\u00b10.62 (0.73--3.19)%",
    "observed_90_coverage_percent": "74.89\u00b111.19 (45.00--89.36)%",
    "mean_pi_width": "1.5639\u00b10.0553",
}
assert supplementary_table_s8.loc[1].to_dict() == {
    "interval_rule": "Locally weighted",
    "neff_condition": r"$n_{\mathrm{eff}}\geq80$",
    "molecules_per_split": "2488.1\u00b170.8 (2375--2636)",
    "test_fraction_percent": "98.37\u00b10.62 (96.81--99.27)%",
    "observed_90_coverage_percent": "89.83\u00b11.06 (86.75--91.47)%",
    "mean_pi_width": "1.4954\u00b10.0434",
}

sensitivity_claims = {
    ("minimum_neff", 40.0): (0.006150, 0.896909),
    ("minimum_neff", 120.0): (0.037843, 0.895638),
    ("bandwidth", 0.60): (0.145796, 0.891953),
    ("bandwidth", 1.20): (0.001801, 0.897693),
}
for (family, value), (expected_fallback, expected_coverage) in sensitivity_claims.items():
    row = sensitivity_summary.loc[
        sensitivity_summary["sensitivity_family"].eq(family)
        & np.isclose(sensitivity_summary["varied_value"], value)
    ].iloc[0]
    assert np.isclose(row["fallback_fraction_mean"], expected_fallback, atol=5e-7)
    assert np.isclose(row["empirical_coverage_mean"], expected_coverage, atol=5e-7)

final_tables = {
    "local_conformal_summary.csv": summary,
    "local_conformal_fallback_summary.csv": fallback_summary,
    "local_conformal_map_width_range_summary.csv": map_range_summary,
    "local_conformal_width_coverage_summary.csv": width_coverage_summary,
    "local_conformal_matched_coverage_width_summary.csv": matched_summary,
    "local_conformal_support_summary.csv": support_summary,
    "local_conformal_sensitivity_summary.csv": sensitivity_summary,
    "local_conformal_coordinate_structure_correlations_summary.csv": correlation_summary,
}
for filename, frame in final_tables.items():
    frame.to_csv(TABLE_ROOT / filename, index=False)
for filename in final_tables:
    path = TABLE_ROOT / filename
    assert path.is_file() and path.stat().st_size > 0, path
print("Final local-conformal summary CSVs written:", ", ".join(sorted(final_tables)))
display(summary.loc[summary["stratum"].eq("all")])


In [ ]:
final_paths = [
    TABLE_ROOT / "local_conformal_summary.csv",
    TABLE_ROOT / "local_conformal_fallback_summary.csv",
    TABLE_ROOT / "local_conformal_map_width_range_summary.csv",
    TABLE_ROOT / "local_conformal_width_coverage_summary.csv",
    TABLE_ROOT / "local_conformal_matched_coverage_width_summary.csv",
    TABLE_ROOT / "local_conformal_support_summary.csv",
    TABLE_ROOT / "local_conformal_sensitivity_summary.csv",
    TABLE_ROOT / "local_conformal_coordinate_structure_correlations_summary.csv",
    FIGURE_ROOT / "local_conformal_width_coverage_curve.png",
]
assert all(path.is_file() and path.stat().st_size > 0 for path in final_paths)
assert len(assignments) == len(similarity) == 75877
assert len(by_split) == 30 * (1 + len(METHODS)) * 5
assert assignments.groupby("split_index").size().eq(
    registry.loc[registry["set"].eq("test")].groupby("split_index").size()
).all()
print("Local conformal final-summary integrity: PASS")


## Article Supplementary Table S5B: endpoint heterogeneity and interval width

This is the executable source for panel B of the **article's** Supplementary
Table S5. The internal `tableS5_paired_model_comparisons.tex` from stage 03 is
a different table. The existing four joint quadrants are also distinct from
the five B_Tox quintiles reported here.

For each of the 30 splits, calculate the 20th, 40th, 60th and 80th percentiles
of B_Tox using **train only** (linear quantile interpolation). Apply these
four boundaries to test molecules; ties at a boundary go to the upper group.
Keep the same stored Plain predictions and the primary four-coordinate local
intervals, including the existing low-support fallback. No model is refitted.

Report sample variances (ddof=1), mean squared error, within-group R-squared,
mean full interval width and width divided by the within-group endpoint
standard deviation. Compute every statistic, including the ratio, within
each split before summarizing its mean and sample SD over the 30 splits.
Coverage is retained in the machine-readable diagnostics alongside widths.

Lower endpoint dispersion is a property the contextual coordinates are
intended to reveal. Interval widths are obtained from calibration errors;
test endpoint variance is a diagnostic, never an input to interval fitting.
The following cells export the final compact CSV used for Supplementary Table S5B.

In [ ]:
# Reuse the corrected cohort, split registry and local intervals from this notebook.
S5B_QUANTILES = np.array([0.2, 0.4, 0.6, 0.8])
s5b_rows = []
assert not assignments.duplicated(["split_index", "ligand_id"]).any()
assert sorted(registry["split_index"].unique()) == list(range(30))

for split_index, test_rows in assignments.groupby("split_index", sort=True):
    split_registry = registry.loc[registry["split_index"].eq(split_index)]
    train_ids = split_registry.loc[split_registry["set"].eq("train"), "ligand_id"].astype(int)
    test_ids = split_registry.loc[split_registry["set"].eq("test"), "ligand_id"].astype(int)
    calibration_ids = split_registry.loc[split_registry["set"].eq("val"), "ligand_id"].astype(int)
    assert set(test_rows["ligand_id"]) == set(test_ids)
    assert set(train_ids).isdisjoint(test_ids) and set(train_ids).isdisjoint(calibration_ids)
    train_btox = by_id.loc[train_ids, "B_Tox"].to_numpy(float)
    assert np.isfinite(train_btox).all()
    boundaries = np.quantile(train_btox, S5B_QUANTILES, method="linear")
    assert np.all(np.diff(boundaries) > 0), (split_index, boundaries)
    test_rows = test_rows.copy()
    np.testing.assert_allclose(test_rows["B_Tox"], by_id.loc[test_rows["ligand_id"], "B_Tox"], rtol=0, atol=1e-12)
    np.testing.assert_allclose(test_rows["y_true"], by_id.loc[test_rows["ligand_id"], TARGET], rtol=0, atol=1e-12)
    test_rows["quintile"] = np.searchsorted(boundaries, test_rows["B_Tox"].to_numpy(float), side="right") + 1
    assert sorted(test_rows["quintile"].unique()) == [1, 2, 3, 4, 5]
    for quintile, group in test_rows.groupby("quintile", sort=True):
        y = group["y_true"].to_numpy(float)
        pred = group["y_pred"].to_numpy(float)
        errors = y - pred
        widths = group["biology_joint_4d_interval_width_90"].to_numpy(float)
        assert len(y) > 1 and np.isfinite(np.column_stack([y, pred, widths])).all()
        endpoint_variance = float(np.var(y, ddof=1))
        assert endpoint_variance > 0, (split_index, quintile)
        endpoint_sd = float(np.sqrt(endpoint_variance))
        mse = float(np.mean(errors ** 2))
        r_squared = safe_r2(y, pred)
        residual_variance = float(np.var(errors, ddof=1))
        # Independent identities catch variance/R-squared and signed-residual mistakes.
        assert np.isclose(r_squared, 1 - len(y) * mse / ((len(y) - 1) * endpoint_variance))
        assert np.isclose(mse, (len(y) - 1) / len(y) * residual_variance + errors.mean() ** 2)
        covered = np.abs(errors) <= widths / 2
        np.testing.assert_array_equal(covered, group["biology_joint_4d_covered_90"].to_numpy(bool))
        s5b_rows.append({
            "split_index": int(split_index), "seed": int(group["seed"].iloc[0]),
            "quintile": int(quintile), "n_train_for_boundaries": len(train_ids),
            **{f"btox_train_q{int(q * 100):02d}": float(edge) for q, edge in zip(S5B_QUANTILES, boundaries)},
            "n_test": len(group), "endpoint_mean": float(y.mean()),
            "endpoint_variance": endpoint_variance, "endpoint_sd": endpoint_sd,
            "residual_variance": residual_variance, "mean_error": float(errors.mean()),
            "mse": mse, "r_squared": r_squared,
            "mean_interval_width": float(widths.mean()),
            "width_to_endpoint_sd": float(widths.mean() / endpoint_sd),
            "covered_count": int(covered.sum()), "empirical_coverage": float(covered.mean()),
            "fallback_count": int(group["biology_joint_4d_fallback"].sum()),
        })

s5b_by_split = pd.DataFrame(s5b_rows)
assert len(s5b_by_split) == 150
assert s5b_by_split.groupby("quintile")["split_index"].nunique().eq(30).all()
for split_index, rows in s5b_by_split.groupby("split_index"):
    source = assignments.loc[assignments["split_index"].eq(split_index)]
    counts = rows["n_test"].to_numpy()
    assert counts.sum() == len(source)
    assert np.isclose(np.average(rows["mse"], weights=counts), np.mean((source.y_true - source.y_pred) ** 2))
    assert np.isclose(np.average(rows["mean_interval_width"], weights=counts), source["biology_joint_4d_interval_width_90"].mean())
    assert rows["covered_count"].sum() == source["biology_joint_4d_covered_90"].sum()
    # Reconstruct total endpoint variance from within- and between-quintile parts.
    total_ss = ((counts - 1) * rows["endpoint_variance"] + counts * (rows["endpoint_mean"] - source.y_true.mean()) ** 2).sum()
    assert np.isclose(total_ss / (len(source) - 1), source.y_true.var(ddof=1))

s5b_metrics = ["n_test", "endpoint_mean", "endpoint_variance", "endpoint_sd", "residual_variance",
               "mean_error", "mse", "r_squared", "mean_interval_width", "width_to_endpoint_sd",
               "covered_count", "empirical_coverage", "fallback_count"]
s5b_summary = summarize(s5b_by_split, ["quintile"], s5b_metrics)
s5b_summary["n_splits"] = 30
# Full precision in the summaries; formatting is shared by CSV and both TeX variants.
s5b_precision = {"n_test": 1, "endpoint_variance": 4, "residual_variance": 4, "mse": 4,
                 "r_squared": 3, "mean_interval_width": 3, "width_to_endpoint_sd": 3}
s5b_display = pd.DataFrame({"quintile": s5b_summary["quintile"]})
for metric, decimals in s5b_precision.items():
    s5b_display[metric] = [
        f"{row[metric + '_mean']:.{decimals}f}±{row[metric + '_std']:.{decimals}f}"
        for _, row in s5b_summary.iterrows()
    ]
display(Markdown("### Article Supplementary Table S5B — mean ± sample SD over 30 splits"))
display(s5b_display)
print("S5B quintile diagnostics: 150 split/group rows; train/test isolation and aggregate identities PASS")

In [ ]:
# The compact display CSV is the final Supplementary Table S5B source.
s5b_display_path = TABLE_ROOT / "tableS5B_btox_quintiles.csv"
s5b_display.to_csv(s5b_display_path, index=False)
pd.testing.assert_frame_equal(pd.read_csv(s5b_display_path), s5b_display)
assert s5b_display_path.is_file() and s5b_display_path.stat().st_size > 0
print("Final Supplementary Table S5B CSV: PASS")


## Current docking-coordinate sensitivity (Supplementary Table S6)

This executable analysis replaces the unregistered historical Table S6 calculation. It does not retrain a point-prediction model. For every registered split it reads the stored Plain-model calibration residuals and held-out predictions, then compares raw $E_{\mathrm{mean}}$, the target-wise standardized 44-score mean, and PC1 of that standardized panel.

Target-wise scaling and PC1 are fitted on calibration molecules only. The tested docking scalar is combined with hERG, Respiratory and Neurotoxicity-DI under the current local-conformal rule ($h=0.90$; fixed global fallback when $n_{\mathrm{eff}}<80$). All coordinate variants use calibration-only scaling and held-out test evaluation.

In [ ]:
# Canonical runner; uses saved predictions and calibration residuals only.
import runpy
runpy.run_path(str(ROOT / 'scripts/run_docking_coordinate_sensitivity.py'), run_name='__main__')

## Interpretation boundary

This audit supports only a direct empirical comparison of fixed interval rules on the corrected repeated-split system. It does **not** prove conditional calibration, causality of the biology coordinates, structural independence of the test set, or prospective regulatory validity. The structural-descriptor and predicted-value rules are fixed diagnostic baselines, not separately optimized competing methods.

In [ ]:
# Artifact preview — final tables and figure generated by this notebook.
from IPython.display import Image as NotebookImage, Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
table_previews = [
    ("Local-conformal 30-split summary", ROOT / "results" / "tables" / "local_conformal_summary.csv"),
    ("B_Tox quintile Supplementary Table S5B", ROOT / "results" / "tables" / "tableS5B_btox_quintiles.csv"),
    ("Docking-coordinate sensitivity (Table S6)", ROOT / "results" / "tables" / "tableS6_docking_coordinate.csv"),
]
for title, path in table_previews:
    assert path.is_file() and path.stat().st_size > 0, path
    display(Markdown(f"## {title}: `{path.relative_to(ROOT)}`"))
    display(pd.read_csv(path))
curve = ROOT / "results" / "figures" / "local_conformal_width_coverage_curve.png"
assert curve.is_file() and curve.stat().st_size > 0, curve
display(Markdown(f"## Fixed width–coverage curve: `{curve.relative_to(ROOT)}`"))
display(NotebookImage(filename=str(curve)))
